# hml_r_std_5m 因子

该因子的构建方法：在过去5个月内，用个股每日最高（低）价除以前一日收盘价计算日内最大涨（跌）幅，此处有复权处理，再计算最大涨（跌）幅序列的标准差，得到high_r_std_5m(low_r_std_5m)因子，用二者相减，得到hml_r_std_5m因子

## 原始因子IC、ICIR、RankIC、RankICIR、因子收益率

In [ ]:
# ============================================================
# hml_r_std_5m 因子测试：IC、RankIC、因子收益率、t值
# 截面周期：30日
# 输出：纵向汇总指标表、IC时间序列图、RankIC时间序列图
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

FACTOR_NAME = "hml_r_std_5m"

LOOKBACK_DAYS = 105       # 近5个月约105个交易日
REBALANCE_DAYS = 30       # 截面周期30日
MIN_OBS = 80              # 计算滚动波动率所需最少观测数
MIN_CROSS_SECTION = 100   # 每期截面最少股票数

WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99


# =========================
# 2. 中文字体设置
# =========================

def set_chinese_font():
    import matplotlib

    font_candidates = [
        "Noto Sans CJK SC",
        "Noto Sans CJK JP",
        "SimHei",
        "Microsoft YaHei",
        "Arial Unicode MS",
        "WenQuanYi Micro Hei",
    ]

    available_fonts = set(f.name for f in matplotlib.font_manager.fontManager.ttflist)

    for font in font_candidates:
        if font in available_fonts:
            plt.rcParams["font.sans-serif"] = [font]
            plt.rcParams["axes.unicode_minus"] = False
            return

    plt.rcParams["axes.unicode_minus"] = False


set_chinese_font()


# =========================
# 3. 读取 A 股日行情数据
# =========================

start_dt = pd.to_datetime(START_DATE)
end_dt = pd.to_datetime(END_DATE)

query_start_date = (start_dt - pd.Timedelta(days=260)).strftime("%Y-%m-%d")
query_end_date = end_dt.strftime("%Y-%m-%d")


def dai_query_df(sql_text):
    result = dai.query(sql_text)
    return result.df()


# cn_stock_bar1d 在部分 BigQuant 环境中没有 is_suspended、is_st、list_date 等字段，
# 因此这里只读取计算 hml_r_std_5m 必需字段，避免字段不存在导致报错。
sql_with_pre_close = f"""
SELECT
    date,
    instrument,
    open,
    high,
    low,
    close,
    pre_close
FROM cn_stock_bar1d
WHERE date >= '{query_start_date}'
  AND date <= '{query_end_date}'
ORDER BY date, instrument
"""

sql_without_pre_close = f"""
SELECT
    date,
    instrument,
    open,
    high,
    low,
    close
FROM cn_stock_bar1d
WHERE date >= '{query_start_date}'
  AND date <= '{query_end_date}'
ORDER BY date, instrument
"""

try:
    bar = dai_query_df(sql_with_pre_close)
except Exception:
    bar = dai_query_df(sql_without_pre_close)

bar["date"] = pd.to_datetime(bar["date"])

for col in ["open", "high", "low", "close", "pre_close"]:
    if col in bar.columns:
        bar[col] = pd.to_numeric(bar[col], errors="coerce")

bar = bar.dropna(subset=["date", "instrument", "open", "high", "low", "close"])
bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)

bar = bar[
    (bar["open"] > 0) &
    (bar["high"] > 0) &
    (bar["low"] > 0) &
    (bar["close"] > 0)
].copy()

bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)


# =========================
# 4. 计算 hml_r_std_5m 因子
# =========================
# high_r = 当日最高价 / 前一交易日收盘价 - 1
# low_r  = 当日最低价 / 前一交易日收盘价 - 1
# high_r_std_5m = 近5个月 high_r 序列标准差
# low_r_std_5m  = 近5个月 low_r 序列标准差
# hml_r_std_5m  = high_r_std_5m - low_r_std_5m

if "pre_close" not in bar.columns:
    bar["pre_close"] = np.nan

bar["pre_close_shift"] = bar.groupby("instrument")["close"].shift(1)
bar["pre_close"] = bar["pre_close"].where(
    bar["pre_close"].notna() & (bar["pre_close"] > 0),
    bar["pre_close_shift"]
)

bar["high_r"] = bar["high"] / bar["pre_close"] - 1
bar["low_r"] = bar["low"] / bar["pre_close"] - 1

bar.loc[~np.isfinite(bar["high_r"]), "high_r"] = np.nan
bar.loc[~np.isfinite(bar["low_r"]), "low_r"] = np.nan

bar["high_r_std_5m"] = (
    bar.groupby("instrument")["high_r"]
    .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
    .std()
    .reset_index(level=0, drop=True)
)

bar["low_r_std_5m"] = (
    bar.groupby("instrument")["low_r"]
    .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
    .std()
    .reset_index(level=0, drop=True)
)

bar[FACTOR_NAME] = bar["high_r_std_5m"] - bar["low_r_std_5m"]


# =========================
# 5. 构造30日持有期未来收益
# =========================

bar["future_close_30d"] = bar.groupby("instrument")["close"].shift(-REBALANCE_DAYS)
bar["future_ret_30d"] = bar["future_close_30d"] / bar["close"] - 1
bar.loc[~np.isfinite(bar["future_ret_30d"]), "future_ret_30d"] = np.nan


# =========================
# 6. 取每30个交易日一个截面
# =========================

trade_dates = (
    bar.loc[
        (bar["date"] >= start_dt) &
        (bar["date"] <= end_dt),
        "date"
    ]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

rebalance_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()

test_df = bar[bar["date"].isin(rebalance_dates)].copy()

test_df = test_df[
    test_df[FACTOR_NAME].notna() &
    test_df["future_ret_30d"].notna()
].copy()


# =========================
# 7. 截面去极值、标准化
# =========================

def winsorize_series(s, q_low=0.01, q_high=0.99):
    low = s.quantile(q_low)
    high = s.quantile(q_high)
    return s.clip(lower=low, upper=high)


def zscore_series(s):
    std = s.std(ddof=1)
    if std == 0 or np.isnan(std):
        return pd.Series(np.nan, index=s.index)
    return (s - s.mean()) / std


test_df["factor_w"] = (
    test_df.groupby("date")[FACTOR_NAME]
    .transform(lambda x: winsorize_series(x, WINSOR_Q_LOW, WINSOR_Q_HIGH))
)

test_df["factor_z"] = (
    test_df.groupby("date")["factor_w"]
    .transform(zscore_series)
)

test_df = test_df[
    test_df["factor_z"].notna() &
    test_df["future_ret_30d"].notna()
].copy()


# =========================
# 8. 单期截面指标计算
# =========================

def calc_cross_section_metrics(g):
    g = g[["factor_z", "future_ret_30d"]].dropna().copy()
    n = len(g)

    if n < MIN_CROSS_SECTION:
        return pd.Series({
            "样本数": n,
            "IC": np.nan,
            "RankIC": np.nan,
            "因子收益率": np.nan,
            "t值": np.nan,
        })

    x = g["factor_z"].values.astype(float)
    y = g["future_ret_30d"].values.astype(float)

    ic = pd.Series(x).corr(pd.Series(y), method="pearson")
    rank_ic = pd.Series(x).corr(pd.Series(y), method="spearman")

    x_mean = np.mean(x)
    y_mean = np.mean(y)
    x_var_sum = np.sum((x - x_mean) ** 2)

    if x_var_sum <= 0:
        beta = np.nan
        t_value = np.nan
    else:
        beta = np.sum((x - x_mean) * (y - y_mean)) / x_var_sum
        residual = y - (y_mean + beta * (x - x_mean))
        sse = np.sum(residual ** 2)
        dof = n - 2

        if dof > 0:
            sigma2 = sse / dof
            se_beta = np.sqrt(sigma2 / x_var_sum)
            t_value = beta / se_beta if se_beta > 0 else np.nan
        else:
            t_value = np.nan

    return pd.Series({
        "样本数": n,
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": beta,
        "t值": t_value,
    })


ts_metrics = (
    test_df.groupby("date")
    .apply(calc_cross_section_metrics)
    .reset_index()
    .rename(columns={"date": "截面日期"})
)

ts_metrics = ts_metrics.dropna(subset=["IC", "RankIC", "因子收益率", "t值"]).copy()


# =========================
# 9. 汇总指标，纵向表格输出
# =========================

def calc_ir(s):
    std = s.std(ddof=1)
    if std == 0 or np.isnan(std):
        return np.nan
    return s.mean() / std


def fmt_value(x):
    if isinstance(x, (int, np.integer)):
        return str(x)
    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return np.nan
        return f"{x:.6f}"
    return x


summary_dict = {
    "因子": FACTOR_NAME,
    "开始日期": START_DATE,
    "结束日期": END_DATE,
    "截面周期": f"{REBALANCE_DAYS}日",
    "有效截面数": int(len(ts_metrics)),
    "平均样本数": ts_metrics["样本数"].mean(),
    "IC均值": ts_metrics["IC"].mean(),
    "ICIR": calc_ir(ts_metrics["IC"]),
    "RankIC均值": ts_metrics["RankIC"].mean(),
    "RankICIR": calc_ir(ts_metrics["RankIC"]),
    "因子收益率均值": ts_metrics["因子收益率"].mean(),
    "t值均值": ts_metrics["t值"].mean(),
}

summary_vertical = pd.DataFrame({
    "指标": list(summary_dict.keys()),
    "数值": [fmt_value(v) for v in summary_dict.values()],
})

display(summary_vertical)


# =========================
# 10. 绘制 IC 时间序列图
# =========================

plt.figure(figsize=(14, 6))
plt.plot(ts_metrics["截面日期"], ts_metrics["IC"], marker="o", linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(ts_metrics["IC"].mean(), linestyle="-", linewidth=1)

plt.title(f"{FACTOR_NAME} 因子 IC 时间序列")
plt.xlabel("截面日期")
plt.ylabel("IC")
plt.grid(True, alpha=0.3)

plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# =========================
# 11. 绘制 RankIC 时间序列图
# =========================

plt.figure(figsize=(14, 6))
plt.plot(ts_metrics["截面日期"], ts_metrics["RankIC"], marker="o", linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(ts_metrics["RankIC"].mean(), linestyle="-", linewidth=1)

plt.title(f"{FACTOR_NAME} 因子 RankIC 时间序列")
plt.xlabel("截面日期")
plt.ylabel("RankIC")
plt.grid(True, alpha=0.3)

plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


hml_r_std_5m 在当前测试区间内表现为较明确的负向因子：IC均值为 -0.0360，RankIC均值为 -0.0805，说明因子值越高，未来30日收益整体越差，且排序层面的区分能力强于线性相关性；同时因子收益率均值为 -0.0035、t值均值为 -2.4777，方向具有一定统计显著性。但从IC与RankIC时间序列看，因子波动较大，存在阶段性失效或反向表现。

## 市值中性化后IC、ICIR、RankIC、RankICIR、因子收益率

In [ ]:
# ============================================================
# hml_r_std_5m 因子测试：市值中性化后 IC、RankIC、因子收益率、t值
# 截面周期：30日
# 输出：纵向指标表、IC时间序列图、RankIC时间序列图
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

RAW_FACTOR_NAME = "hml_r_std_5m"
FACTOR_NAME = "hml_r_std_5m_mkt_neutral"

LOOKBACK_DAYS = 105       # 近5个月约105个交易日
REBALANCE_DAYS = 30       # 截面周期30日
MIN_OBS = 80              # 计算滚动波动率所需最少观测数
MIN_CROSS_SECTION = 100   # 每期截面最少股票数

WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")


# =========================
# 2. 中文字体设置
# =========================

def set_chinese_font():
    import matplotlib
    font_candidates = [
        "Noto Sans CJK SC",
        "Noto Sans CJK JP",
        "SimHei",
        "Microsoft YaHei",
        "Arial Unicode MS",
        "WenQuanYi Micro Hei",
    ]

    available_fonts = set(f.name for f in matplotlib.font_manager.fontManager.ttflist)

    for font in font_candidates:
        if font in available_fonts:
            plt.rcParams["font.sans-serif"] = [font]
            plt.rcParams["axes.unicode_minus"] = False
            return

    plt.rcParams["axes.unicode_minus"] = False

set_chinese_font()


# =========================
# 3. 通用查询函数
# =========================

def query_first_success(sql_list):
    last_error = None
    for sql in sql_list:
        try:
            df = dai.query(sql).df()
            if df is not None and len(df) > 0:
                return df
        except Exception as e:
            last_error = e
    if last_error is not None:
        raise last_error
    raise ValueError("所有查询均未返回有效数据。")


def query_optional(sql_list):
    for sql in sql_list:
        try:
            df = dai.query(sql).df()
            if df is not None and len(df) > 0:
                return df
        except Exception:
            continue
    return None


# =========================
# 4. 读取 A 股日行情数据
# =========================

query_start_date = (
    pd.to_datetime(START_DATE) - pd.Timedelta(days=260)
).strftime("%Y-%m-%d")

bar_sql_list = [
    f"""
    SELECT
        date,
        instrument,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume,
        turn
    FROM cn_stock_bar1d
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT
        date,
        instrument,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume
    FROM cn_stock_bar1d
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT
        date,
        instrument,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        turn
    FROM cn_stock_bar1d
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT
        date,
        instrument,
        open,
        high,
        low,
        close,
        amount,
        volume
    FROM cn_stock_bar1d
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
]

bar = query_first_success(bar_sql_list)

bar["date"] = pd.to_datetime(bar["date"])

for col in ["open", "high", "low", "close", "pre_close", "amount", "volume", "turn"]:
    if col in bar.columns:
        bar[col] = pd.to_numeric(bar[col], errors="coerce")

for col in ["pre_close", "amount", "volume", "turn"]:
    if col not in bar.columns:
        bar[col] = np.nan

bar = bar.dropna(subset=["date", "instrument", "open", "high", "low", "close"])
bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)

base_filter = (
    (bar["open"] > 0) &
    (bar["high"] > 0) &
    (bar["low"] > 0) &
    (bar["close"] > 0)
)

if bar["amount"].notna().any():
    base_filter = base_filter & (bar["amount"].fillna(0) > 0)

if bar["volume"].notna().any():
    base_filter = base_filter & (bar["volume"].fillna(0) > 0)

bar = bar[base_filter].copy()
bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)


# =========================
# 5. 读取或构造市值数据
# =========================

mkt_sql_list = [
    f"""
    SELECT date, instrument, market_cap AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_market_cap AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, float_market_cap AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_mv AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, circ_mv AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, market_cap AS mkt_cap
    FROM cn_stock_factors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_market_cap AS mkt_cap
    FROM cn_stock_factors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, float_market_cap AS mkt_cap
    FROM cn_stock_factors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_mv AS mkt_cap
    FROM cn_stock_daily_basic
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, circ_mv AS mkt_cap
    FROM cn_stock_daily_basic
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_mv AS mkt_cap
    FROM cn_stock_valuation
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, circ_mv AS mkt_cap
    FROM cn_stock_valuation
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_market_cap AS mkt_cap
    FROM cn_stock_valuation
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, float_market_cap AS mkt_cap
    FROM cn_stock_valuation
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
]

mkt = query_optional(mkt_sql_list)

if mkt is not None:
    mkt["date"] = pd.to_datetime(mkt["date"])
    mkt["mkt_cap"] = pd.to_numeric(mkt["mkt_cap"], errors="coerce")
    mkt = mkt.dropna(subset=["date", "instrument", "mkt_cap"])
    mkt = mkt[mkt["mkt_cap"] > 0].copy()
    mkt = mkt.drop_duplicates(["date", "instrument"])
    bar = bar.merge(mkt[["date", "instrument", "mkt_cap"]], on=["date", "instrument"], how="left")
else:
    if ("amount" in bar.columns) and ("turn" in bar.columns) and bar["amount"].notna().any() and bar["turn"].notna().any():
        turn_abs = bar["turn"].abs()
        turn_frac = np.where(turn_abs > 1, turn_abs / 100.0, turn_abs)
        bar["mkt_cap"] = bar["amount"] / turn_frac
        bar.loc[~np.isfinite(bar["mkt_cap"]), "mkt_cap"] = np.nan
    else:
        raise ValueError(
            "未能读取市值字段，也无法通过 amount/turn 构造市值代理。"
            "请在 mkt_sql_list 中补充当前 BigQuant 环境可用的市值表和字段。"
        )

bar["mkt_cap"] = pd.to_numeric(bar["mkt_cap"], errors="coerce")
bar.loc[~np.isfinite(bar["mkt_cap"]), "mkt_cap"] = np.nan
bar.loc[bar["mkt_cap"] <= 0, "mkt_cap"] = np.nan
bar["log_mkt_cap"] = np.log(bar["mkt_cap"])


# =========================
# 6. 计算 hml_r_std_5m 原始因子
# =========================
# high_r = 当日最高价 / 前一交易日收盘价 - 1
# low_r  = 当日最低价 / 前一交易日收盘价 - 1
# high_r_std_5m = 近5个月 high_r 序列标准差
# low_r_std_5m  = 近5个月 low_r 序列标准差
# hml_r_std_5m  = high_r_std_5m - low_r_std_5m

if bar["pre_close"].isna().all():
    bar["pre_close_calc"] = bar.groupby("instrument")["close"].shift(1)
    pre_close_used = bar["pre_close_calc"]
else:
    bar["pre_close_calc"] = bar.groupby("instrument")["close"].shift(1)
    pre_close_used = bar["pre_close"].fillna(bar["pre_close_calc"])

bar["high_r"] = bar["high"] / pre_close_used - 1
bar["low_r"] = bar["low"] / pre_close_used - 1

bar.loc[~np.isfinite(bar["high_r"]), "high_r"] = np.nan
bar.loc[~np.isfinite(bar["low_r"]), "low_r"] = np.nan

bar["high_r_std_5m"] = (
    bar.groupby("instrument")["high_r"]
    .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
    .std()
    .reset_index(level=0, drop=True)
)

bar["low_r_std_5m"] = (
    bar.groupby("instrument")["low_r"]
    .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
    .std()
    .reset_index(level=0, drop=True)
)

bar[RAW_FACTOR_NAME] = bar["high_r_std_5m"] - bar["low_r_std_5m"]


# =========================
# 7. 构造30日持有期未来收益
# =========================

bar["future_close_30d"] = bar.groupby("instrument")["close"].shift(-REBALANCE_DAYS)
bar["future_ret_30d"] = bar["future_close_30d"] / bar["close"] - 1
bar.loc[~np.isfinite(bar["future_ret_30d"]), "future_ret_30d"] = np.nan


# =========================
# 8. 取每30个交易日一个截面
# =========================

trade_dates = (
    bar.loc[
        (bar["date"] >= pd.to_datetime(START_DATE)) &
        (bar["date"] <= pd.to_datetime(END_DATE)),
        "date"
    ]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

rebalance_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()

test_df = bar[bar["date"].isin(rebalance_dates)].copy()

test_df = test_df[
    test_df[RAW_FACTOR_NAME].notna() &
    test_df["log_mkt_cap"].notna() &
    test_df["future_ret_30d"].notna()
].copy()


# =========================
# 9. 截面去极值、市值中性化、标准化
# =========================

def winsorize_series(s, q_low=0.01, q_high=0.99):
    low = s.quantile(q_low)
    high = s.quantile(q_high)
    return s.clip(lower=low, upper=high)


def zscore_series(s):
    std = s.std(ddof=1)
    if std == 0 or np.isnan(std):
        return pd.Series(np.nan, index=s.index)
    return (s - s.mean()) / std


def neutralize_by_log_mkt_cap(g):
    tmp = g[["factor_w", "log_mkt_cap"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    out = pd.Series(np.nan, index=g.index)

    if len(tmp) < MIN_CROSS_SECTION:
        return out

    if tmp["log_mkt_cap"].std(ddof=1) == 0 or np.isnan(tmp["log_mkt_cap"].std(ddof=1)):
        return out

    y = tmp["factor_w"].values.astype(float)
    x = tmp["log_mkt_cap"].values.astype(float)

    X = np.column_stack([np.ones(len(tmp)), x])

    try:
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        residual = y - X @ beta
        out.loc[tmp.index] = residual
    except Exception:
        pass

    return out


test_df["factor_w"] = (
    test_df.groupby("date")[RAW_FACTOR_NAME]
    .transform(lambda x: winsorize_series(x, WINSOR_Q_LOW, WINSOR_Q_HIGH))
)

test_df["factor_neutral"] = (
    test_df.groupby("date", group_keys=False)
    .apply(neutralize_by_log_mkt_cap)
)

test_df["factor_z"] = (
    test_df.groupby("date")["factor_neutral"]
    .transform(zscore_series)
)

test_df = test_df[
    test_df["factor_z"].notna() &
    test_df["future_ret_30d"].notna()
].copy()


# =========================
# 10. 单期截面指标计算
# =========================

def calc_cross_section_metrics(g):
    g = g[["factor_z", "future_ret_30d"]].dropna().copy()
    n = len(g)

    if n < MIN_CROSS_SECTION:
        return pd.Series({
            "样本数": n,
            "IC": np.nan,
            "RankIC": np.nan,
            "因子收益率": np.nan,
            "t值": np.nan,
        })

    x = g["factor_z"].values.astype(float)
    y = g["future_ret_30d"].values.astype(float)

    x_var = np.sum((x - np.mean(x)) ** 2)
    if x_var == 0 or np.isnan(x_var):
        return pd.Series({
            "样本数": n,
            "IC": np.nan,
            "RankIC": np.nan,
            "因子收益率": np.nan,
            "t值": np.nan,
        })

    ic = pd.Series(x).corr(pd.Series(y), method="pearson")
    rank_ic = pd.Series(x).corr(pd.Series(y), method="spearman")

    x_mean = np.mean(x)
    y_mean = np.mean(y)

    beta = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)

    residual = y - (y_mean + beta * (x - x_mean))
    sse = np.sum(residual ** 2)

    dof = n - 2
    if dof > 0:
        sigma2 = sse / dof
        se_beta = np.sqrt(sigma2 / np.sum((x - x_mean) ** 2))
        t_value = beta / se_beta if se_beta > 0 else np.nan
    else:
        t_value = np.nan

    return pd.Series({
        "样本数": n,
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": beta,
        "t值": t_value,
    })


ts_metrics = (
    test_df.groupby("date")
    .apply(calc_cross_section_metrics)
    .reset_index()
    .rename(columns={"date": "截面日期"})
)

ts_metrics = ts_metrics.dropna(subset=["IC", "RankIC", "因子收益率", "t值"]).copy()


# =========================
# 11. 汇总指标，纵向输出
# =========================

def calc_ir(s):
    std = s.std(ddof=1)
    if std == 0 or np.isnan(std):
        return np.nan
    return s.mean() / std

summary_vertical = pd.DataFrame({
    "指标": [
        "因子",
        "中性化",
        "开始日期",
        "结束日期",
        "截面周期",
        "有效截面数",
        "平均样本数",
        "IC均值",
        "ICIR",
        "RankIC均值",
        "RankICIR",
        "因子收益率均值",
        "t值均值",
    ],
    "数值": [
        FACTOR_NAME,
        "市值中性化",
        START_DATE,
        END_DATE,
        f"{REBALANCE_DAYS}日",
        int(len(ts_metrics)),
        round(ts_metrics["样本数"].mean(), 6),
        round(ts_metrics["IC"].mean(), 6),
        round(calc_ir(ts_metrics["IC"]), 6),
        round(ts_metrics["RankIC"].mean(), 6),
        round(calc_ir(ts_metrics["RankIC"]), 6),
        round(ts_metrics["因子收益率"].mean(), 6),
        round(ts_metrics["t值"].mean(), 6),
    ]
})

try:
    display(summary_vertical)
except NameError:
    print(summary_vertical.to_string(index=False))


# =========================
# 12. 绘制 IC 时间序列图
# =========================

plt.figure(figsize=(14, 6))
plt.plot(ts_metrics["截面日期"], ts_metrics["IC"], marker="o", linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(ts_metrics["IC"].mean(), linestyle="-", linewidth=1)

plt.title(f"{FACTOR_NAME} 因子 IC 时间序列")
plt.xlabel("截面日期")
plt.ylabel("IC")
plt.grid(True, alpha=0.3)

plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# =========================
# 13. 绘制 RankIC 时间序列图
# =========================

plt.figure(figsize=(14, 6))
plt.plot(ts_metrics["截面日期"], ts_metrics["RankIC"], marker="o", linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(ts_metrics["RankIC"].mean(), linestyle="-", linewidth=1)

plt.title(f"{FACTOR_NAME} 因子 RankIC 时间序列")
plt.xlabel("截面日期")
plt.ylabel("RankIC")
plt.grid(True, alpha=0.3)

plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


## 行业市值中性化后IC、ICIR、RankIC、RankICIR、因子收益率

In [ ]:
# ============================================================
# hml_r_std_5m 因子测试：市值 + 行业中性化后 IC、RankIC、因子收益率、t值
# 截面周期：30日
# 输出：纵向指标表、IC时间序列图、RankIC时间序列图
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

RAW_FACTOR_NAME = "hml_r_std_5m"
FACTOR_NAME = "hml_r_std_5m_mkt_ind_neutral"

LOOKBACK_DAYS = 105       # 近5个月约105个交易日
REBALANCE_DAYS = 30       # 截面周期30日
MIN_OBS = 80              # 计算滚动波动率所需最少观测数
MIN_CROSS_SECTION = 100   # 每期截面最少股票数

WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")


# =========================
# 2. 中文字体设置
# =========================

def set_chinese_font():
    import matplotlib
    font_candidates = [
        "Noto Sans CJK SC",
        "Noto Sans CJK JP",
        "SimHei",
        "Microsoft YaHei",
        "Arial Unicode MS",
        "WenQuanYi Micro Hei",
    ]

    available_fonts = set(f.name for f in matplotlib.font_manager.fontManager.ttflist)

    for font in font_candidates:
        if font in available_fonts:
            plt.rcParams["font.sans-serif"] = [font]
            plt.rcParams["axes.unicode_minus"] = False
            return

    plt.rcParams["axes.unicode_minus"] = False

set_chinese_font()


# =========================
# 3. 通用查询函数
# =========================

def query_first_success(sql_list):
    last_error = None
    for sql in sql_list:
        try:
            df = dai.query(sql).df()
            if df is not None and len(df) > 0:
                return df
        except Exception as e:
            last_error = e
    if last_error is not None:
        raise last_error
    raise ValueError("所有查询均未返回有效数据。")


def query_optional(sql_list):
    for sql in sql_list:
        try:
            df = dai.query(sql).df()
            if df is not None and len(df) > 0:
                return df
        except Exception:
            continue
    return None


def get_table_sample(table_name):
    try:
        return dai.query(f"SELECT * FROM {table_name} LIMIT 1").df()
    except Exception:
        return None


def find_col_case_insensitive(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None


# =========================
# 4. 读取 A 股日行情数据
# =========================

query_start_date = (
    pd.to_datetime(START_DATE) - pd.Timedelta(days=260)
).strftime("%Y-%m-%d")

bar_sql_list = [
    f"""
    SELECT
        date,
        instrument,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume,
        turn
    FROM cn_stock_bar1d
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT
        date,
        instrument,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume
    FROM cn_stock_bar1d
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT
        date,
        instrument,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        turn
    FROM cn_stock_bar1d
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT
        date,
        instrument,
        open,
        high,
        low,
        close,
        amount,
        volume
    FROM cn_stock_bar1d
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
]

bar = query_first_success(bar_sql_list)

bar["date"] = pd.to_datetime(bar["date"])

for col in ["open", "high", "low", "close", "pre_close", "amount", "volume", "turn"]:
    if col in bar.columns:
        bar[col] = pd.to_numeric(bar[col], errors="coerce")

for col in ["pre_close", "amount", "volume", "turn"]:
    if col not in bar.columns:
        bar[col] = np.nan

bar = bar.dropna(subset=["date", "instrument", "open", "high", "low", "close"])
bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)

base_filter = (
    (bar["open"] > 0) &
    (bar["high"] > 0) &
    (bar["low"] > 0) &
    (bar["close"] > 0)
)

if bar["amount"].notna().any():
    base_filter = base_filter & (bar["amount"].fillna(0) > 0)

if bar["volume"].notna().any():
    base_filter = base_filter & (bar["volume"].fillna(0) > 0)

bar = bar[base_filter].copy()
bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)


# =========================
# 5. 读取或构造市值数据
# =========================

mkt_sql_list = [
    f"""
    SELECT date, instrument, market_cap AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_market_cap AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, float_market_cap AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_mv AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, circ_mv AS mkt_cap
    FROM cn_stock_prefactors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, market_cap AS mkt_cap
    FROM cn_stock_factors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_market_cap AS mkt_cap
    FROM cn_stock_factors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, float_market_cap AS mkt_cap
    FROM cn_stock_factors
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_mv AS mkt_cap
    FROM cn_stock_daily_basic
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, circ_mv AS mkt_cap
    FROM cn_stock_daily_basic
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_mv AS mkt_cap
    FROM cn_stock_valuation
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, circ_mv AS mkt_cap
    FROM cn_stock_valuation
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, total_market_cap AS mkt_cap
    FROM cn_stock_valuation
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
    f"""
    SELECT date, instrument, float_market_cap AS mkt_cap
    FROM cn_stock_valuation
    WHERE date >= '{query_start_date}'
      AND date <= '{END_DATE}'
    ORDER BY date, instrument
    """,
]

mkt = query_optional(mkt_sql_list)

if mkt is not None:
    mkt["date"] = pd.to_datetime(mkt["date"])
    mkt["mkt_cap"] = pd.to_numeric(mkt["mkt_cap"], errors="coerce")
    mkt = mkt.dropna(subset=["date", "instrument", "mkt_cap"])
    mkt = mkt[mkt["mkt_cap"] > 0].copy()
    mkt = mkt.drop_duplicates(["date", "instrument"])
    bar = bar.merge(mkt[["date", "instrument", "mkt_cap"]], on=["date", "instrument"], how="left")
else:
    if ("amount" in bar.columns) and ("turn" in bar.columns) and bar["amount"].notna().any() and bar["turn"].notna().any():
        turn_abs = bar["turn"].abs()
        turn_frac = np.where(turn_abs > 1, turn_abs / 100.0, turn_abs)
        bar["mkt_cap"] = bar["amount"] / turn_frac
        bar.loc[~np.isfinite(bar["mkt_cap"]), "mkt_cap"] = np.nan
    else:
        raise ValueError(
            "未能读取市值字段，也无法通过 amount/turn 构造市值代理。"
            "请在 mkt_sql_list 中补充当前 BigQuant 环境可用的市值表和字段。"
        )

bar["mkt_cap"] = pd.to_numeric(bar["mkt_cap"], errors="coerce")
bar.loc[~np.isfinite(bar["mkt_cap"]), "mkt_cap"] = np.nan
bar.loc[bar["mkt_cap"] <= 0, "mkt_cap"] = np.nan
bar["log_mkt_cap"] = np.log(bar["mkt_cap"])


# =========================
# 6. 读取行业数据
# =========================

def load_industry_data(start_date, end_date):
    candidate_tables = [
        "cn_stock_industry_component",
        "cn_stock_industry",
        "cn_stock_industry_sw",
        "cn_stock_sw_industry",
        "cn_stock_industry_classification",
        "cn_stock_basic_info",
        "cn_stock_instruments",
        "cn_stock_prefactors",
        "cn_stock_factors",
    ]

    instrument_candidates = ["instrument", "code", "symbol", "security_code", "ts_code"]
    date_candidates = ["date", "trade_date", "update_date", "effective_date", "entry_date"]
    industry_candidates = [
        "sw2021_level1_name",
        "sw2021_level1",
        "sw_level1_name",
        "sw_level1",
        "sw_l1_name",
        "sw_l1",
        "industry_sw_level1_name",
        "industry_sw_level1",
        "industry_level1_name",
        "industry_level1",
        "level1_industry_name",
        "level1_industry",
        "citics_level1_name",
        "citics_level1",
        "industry_name",
        "industry_code",
        "industry",
        "sector",
    ]

    for table in candidate_tables:
        sample = get_table_sample(table)
        if sample is None or sample.empty:
            continue

        columns = list(sample.columns)
        instrument_col = find_col_case_insensitive(columns, instrument_candidates)
        industry_col = find_col_case_insensitive(columns, industry_candidates)
        date_col = find_col_case_insensitive(columns, date_candidates)

        if instrument_col is None or industry_col is None:
            continue

        if date_col is not None:
            sql = f"""
            SELECT
                {date_col} AS date,
                {instrument_col} AS instrument,
                {industry_col} AS industry
            FROM {table}
            WHERE {date_col} >= '{start_date}'
              AND {date_col} <= '{end_date}'
            """
        else:
            sql = f"""
            SELECT
                {instrument_col} AS instrument,
                {industry_col} AS industry
            FROM {table}
            """

        try:
            industry = dai.query(sql).df()
        except Exception:
            continue

        if industry is None or industry.empty:
            continue

        industry["instrument"] = industry["instrument"].astype(str)
        industry["industry"] = industry["industry"].astype(str)
        industry = industry.replace({"industry": {"nan": np.nan, "None": np.nan, "": np.nan}})
        industry = industry.dropna(subset=["instrument", "industry"])

        if "date" in industry.columns:
            industry["date"] = pd.to_datetime(industry["date"], errors="coerce")
            industry = industry.dropna(subset=["date"])
            industry = industry.drop_duplicates(["date", "instrument"], keep="last")
        else:
            industry = industry.drop_duplicates(["instrument"], keep="last")

        if len(industry) > 0 and industry["industry"].nunique() >= 2:
            return industry, table, industry_col

    return None, None, None


industry, industry_table_used, industry_col_used = load_industry_data(query_start_date, END_DATE)

if industry is None:
    raise ValueError(
        "未能自动读取行业字段。请在 load_industry_data 函数中补充当前 BigQuant 环境可用的行业表和字段。"
    )


# =========================
# 7. 计算 hml_r_std_5m 原始因子
# =========================
# high_r = 当日最高价 / 前一交易日收盘价 - 1
# low_r  = 当日最低价 / 前一交易日收盘价 - 1
# high_r_std_5m = 近5个月 high_r 序列标准差
# low_r_std_5m  = 近5个月 low_r 序列标准差
# hml_r_std_5m  = high_r_std_5m - low_r_std_5m

if bar["pre_close"].isna().all():
    bar["pre_close_calc"] = bar.groupby("instrument")["close"].shift(1)
    pre_close_used = bar["pre_close_calc"]
else:
    bar["pre_close_calc"] = bar.groupby("instrument")["close"].shift(1)
    pre_close_used = bar["pre_close"].fillna(bar["pre_close_calc"])

bar["high_r"] = bar["high"] / pre_close_used - 1
bar["low_r"] = bar["low"] / pre_close_used - 1

bar.loc[~np.isfinite(bar["high_r"]), "high_r"] = np.nan
bar.loc[~np.isfinite(bar["low_r"]), "low_r"] = np.nan

bar["high_r_std_5m"] = (
    bar.groupby("instrument")["high_r"]
    .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
    .std()
    .reset_index(level=0, drop=True)
)

bar["low_r_std_5m"] = (
    bar.groupby("instrument")["low_r"]
    .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
    .std()
    .reset_index(level=0, drop=True)
)

bar[RAW_FACTOR_NAME] = bar["high_r_std_5m"] - bar["low_r_std_5m"]


# =========================
# 8. 构造30日持有期未来收益
# =========================

bar["future_close_30d"] = bar.groupby("instrument")["close"].shift(-REBALANCE_DAYS)
bar["future_ret_30d"] = bar["future_close_30d"] / bar["close"] - 1
bar.loc[~np.isfinite(bar["future_ret_30d"]), "future_ret_30d"] = np.nan


# =========================
# 9. 取每30个交易日一个截面
# =========================

trade_dates = (
    bar.loc[
        (bar["date"] >= pd.to_datetime(START_DATE)) &
        (bar["date"] <= pd.to_datetime(END_DATE)),
        "date"
    ]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

rebalance_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()

test_df = bar[bar["date"].isin(rebalance_dates)].copy()


def attach_industry(test_df, industry):
    test_df = test_df.copy()
    test_df["instrument"] = test_df["instrument"].astype(str)

    if "date" in industry.columns:
        industry = industry.copy()
        industry["instrument"] = industry["instrument"].astype(str)
        industry["date"] = pd.to_datetime(industry["date"])
        industry = industry.sort_values(["date", "instrument"])

        exact = test_df.merge(
            industry[["date", "instrument", "industry"]],
            on=["date", "instrument"],
            how="left"
        )

        exact_match_ratio = exact["industry"].notna().mean()
        if exact_match_ratio >= 0.70:
            return exact

        left = test_df.sort_values(["date", "instrument"]).reset_index(drop=True)
        right = industry[["date", "instrument", "industry"]].sort_values(["date", "instrument"]).reset_index(drop=True)

        try:
            asof = pd.merge_asof(
                left,
                right,
                on="date",
                by="instrument",
                direction="backward",
                allow_exact_matches=True
            )
            if asof["industry"].notna().mean() > exact_match_ratio:
                return asof
        except Exception:
            pass

        return exact

    industry = industry.copy()
    industry["instrument"] = industry["instrument"].astype(str)
    industry = industry.drop_duplicates(["instrument"], keep="last")

    return test_df.merge(
        industry[["instrument", "industry"]],
        on="instrument",
        how="left"
    )


test_df = attach_industry(test_df, industry)

test_df = test_df[
    test_df[RAW_FACTOR_NAME].notna() &
    test_df["log_mkt_cap"].notna() &
    test_df["industry"].notna() &
    test_df["future_ret_30d"].notna()
].copy()


# =========================
# 10. 截面去极值、市值 + 行业中性化、标准化
# =========================

def winsorize_series(s, q_low=0.01, q_high=0.99):
    low = s.quantile(q_low)
    high = s.quantile(q_high)
    return s.clip(lower=low, upper=high)


def zscore_series(s):
    std = s.std(ddof=1)
    if std == 0 or np.isnan(std):
        return pd.Series(np.nan, index=s.index)
    return (s - s.mean()) / std


def neutralize_by_log_mkt_cap_and_industry(g):
    tmp = g[["factor_w", "log_mkt_cap", "industry"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    out = pd.Series(np.nan, index=g.index)

    if len(tmp) < MIN_CROSS_SECTION:
        return out

    if tmp["log_mkt_cap"].std(ddof=1) == 0 or np.isnan(tmp["log_mkt_cap"].std(ddof=1)):
        return out

    y = tmp["factor_w"].astype(float)
    log_mkt = tmp["log_mkt_cap"].astype(float)
    log_mkt = log_mkt - log_mkt.mean()

    industry_dummies = pd.get_dummies(tmp["industry"].astype(str), prefix="ind", drop_first=True, dtype=float)

    X = pd.concat([
        pd.Series(1.0, index=tmp.index, name="const"),
        log_mkt.rename("log_mkt_cap"),
        industry_dummies,
    ], axis=1)

    valid_cols = []
    for col in X.columns:
        col_std = X[col].std(ddof=1)
        if col == "const" or (col_std > 0 and np.isfinite(col_std)):
            valid_cols.append(col)
    X = X[valid_cols]

    if len(tmp) <= X.shape[1] + 2:
        return out

    try:
        beta = np.linalg.lstsq(X.values.astype(float), y.values.astype(float), rcond=None)[0]
        residual = y.values.astype(float) - X.values.astype(float) @ beta
        out.loc[tmp.index] = residual
    except Exception:
        pass

    return out


test_df["factor_w"] = (
    test_df.groupby("date")[RAW_FACTOR_NAME]
    .transform(lambda x: winsorize_series(x, WINSOR_Q_LOW, WINSOR_Q_HIGH))
)

test_df["factor_neutral"] = (
    test_df.groupby("date", group_keys=False)
    .apply(neutralize_by_log_mkt_cap_and_industry)
)

test_df["factor_z"] = (
    test_df.groupby("date")["factor_neutral"]
    .transform(zscore_series)
)

test_df = test_df[
    test_df["factor_z"].notna() &
    test_df["future_ret_30d"].notna()
].copy()


# =========================
# 11. 单期截面指标计算
# =========================

def calc_cross_section_metrics(g):
    g = g[["factor_z", "future_ret_30d"]].dropna().copy()
    n = len(g)

    if n < MIN_CROSS_SECTION:
        return pd.Series({
            "样本数": n,
            "IC": np.nan,
            "RankIC": np.nan,
            "因子收益率": np.nan,
            "t值": np.nan,
        })

    x = g["factor_z"].values.astype(float)
    y = g["future_ret_30d"].values.astype(float)

    x_var = np.sum((x - np.mean(x)) ** 2)
    if x_var == 0 or np.isnan(x_var):
        return pd.Series({
            "样本数": n,
            "IC": np.nan,
            "RankIC": np.nan,
            "因子收益率": np.nan,
            "t值": np.nan,
        })

    ic = pd.Series(x).corr(pd.Series(y), method="pearson")
    rank_ic = pd.Series(x).corr(pd.Series(y), method="spearman")

    x_mean = np.mean(x)
    y_mean = np.mean(y)

    beta = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)

    residual = y - (y_mean + beta * (x - x_mean))
    sse = np.sum(residual ** 2)

    dof = n - 2
    if dof > 0:
        sigma2 = sse / dof
        se_beta = np.sqrt(sigma2 / np.sum((x - x_mean) ** 2))
        t_value = beta / se_beta if se_beta > 0 else np.nan
    else:
        t_value = np.nan

    return pd.Series({
        "样本数": n,
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": beta,
        "t值": t_value,
    })


ts_metrics = (
    test_df.groupby("date")
    .apply(calc_cross_section_metrics)
    .reset_index()
    .rename(columns={"date": "截面日期"})
)

ts_metrics = ts_metrics.dropna(subset=["IC", "RankIC", "因子收益率", "t值"]).copy()


# =========================
# 12. 汇总指标，纵向输出
# =========================

def calc_ir(s):
    std = s.std(ddof=1)
    if std == 0 or np.isnan(std):
        return np.nan
    return s.mean() / std

summary_vertical = pd.DataFrame({
    "指标": [
        "因子",
        "中性化",
        "行业来源表",
        "行业字段",
        "开始日期",
        "结束日期",
        "截面周期",
        "有效截面数",
        "平均样本数",
        "IC均值",
        "ICIR",
        "RankIC均值",
        "RankICIR",
        "因子收益率均值",
        "t值均值",
    ],
    "数值": [
        FACTOR_NAME,
        "市值 + 行业中性化",
        industry_table_used,
        industry_col_used,
        START_DATE,
        END_DATE,
        f"{REBALANCE_DAYS}日",
        int(len(ts_metrics)),
        round(ts_metrics["样本数"].mean(), 6),
        round(ts_metrics["IC"].mean(), 6),
        round(calc_ir(ts_metrics["IC"]), 6),
        round(ts_metrics["RankIC"].mean(), 6),
        round(calc_ir(ts_metrics["RankIC"]), 6),
        round(ts_metrics["因子收益率"].mean(), 6),
        round(ts_metrics["t值"].mean(), 6),
    ]
})

try:
    display(summary_vertical)
except NameError:
    print(summary_vertical.to_string(index=False))


# =========================
# 13. 绘制 IC 时间序列图
# =========================

plt.figure(figsize=(14, 6))
plt.plot(ts_metrics["截面日期"], ts_metrics["IC"], marker="o", linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(ts_metrics["IC"].mean(), linestyle="-", linewidth=1)

plt.title(f"{FACTOR_NAME} 因子 IC 时间序列")
plt.xlabel("截面日期")
plt.ylabel("IC")
plt.grid(True, alpha=0.3)

plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# =========================
# 14. 绘制 RankIC 时间序列图
# =========================

plt.figure(figsize=(14, 6))
plt.plot(ts_metrics["截面日期"], ts_metrics["RankIC"], marker="o", linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(ts_metrics["RankIC"].mean(), linestyle="-", linewidth=1)

plt.title(f"{FACTOR_NAME} 因子 RankIC 时间序列")
plt.xlabel("截面日期")
plt.ylabel("RankIC")
plt.grid(True, alpha=0.3)

plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


该因子在做完市值行业中性化后RankICIR有较为明显的提升，在选股排序的稳定性上变得更好了，但IC值仍然不佳，考虑到研报当中显示的，该因子在小市值的回测中有比较好的结果，于是现在考虑对该因子在不同市值区间的指标进行计算

## 分市值计算行业市值中性化IC、ICIR、RankIC、RankICIR、因子收益率

In [ ]:
# ============================================================
# hml_r_std_5m 因子测试：市值行业中性化 + 15档市值分组 + FWL提速省内存版
# 输出：不同市值分组下 IC、ICIR、RankIC、RankICIR、因子收益率均值、t值均值
# 截面周期：30日
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import gc
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

FACTOR_NAME = "hml_r_std_5m"
NEUTRAL_FACTOR_NAME = "factor_mkt_ind_neutral"

LOOKBACK_DAYS = 105       # 近5个月约105个交易日
REBALANCE_DAYS = 30       # 截面周期30日
MIN_OBS = 80              # 计算滚动波动率所需最少观测数
MIN_CROSS_SECTION = 50    # 每个市值分组每期截面最少股票数
N_SIZE_GROUPS = 15        # 市值分组数量，1=小市值，15=大市值

WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99

USE_SQL_FACTOR_CALC = True  # 优先在 DAI SQL 侧完成滚动因子与未来收益计算，显著减少 Python 内存占用

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")
QUERY_START_DATE = (pd.to_datetime(START_DATE) - pd.Timedelta(days=260)).strftime("%Y-%m-%d")


# =========================
# 2. 工具函数
# =========================

_T0 = time.time()


def _elapsed():
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg):
    print(f"[{_elapsed()}] {msg}", flush=True)


def mem_mb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1024 / 1024
    except Exception:
        return np.nan


def progress_rows(name, df):
    m = mem_mb()
    if np.isfinite(m):
        progress(f"{name}：{len(df):,} 行，内存约 {m:.1f} MB")
    else:
        progress(f"{name}：{len(df):,} 行")


def query_df(sql):
    return dai.query(sql).df()


def try_query(sql):
    try:
        df = query_df(sql)
        if df is not None and len(df) > 0:
            return df
    except Exception:
        return None
    return None


def first_success_query(sql_list, err_msg):
    last_error = None
    for i, sql in enumerate(sql_list, 1):
        try:
            df = query_df(sql)
            if df is not None and len(df) > 0:
                return df
        except Exception as e:
            last_error = e
            continue
    raise ValueError(f"{err_msg}。最后一次错误：{last_error}")


def date_in_sql(dates):
    return ", ".join([f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates])


def downcast_float(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def winsorize_array(x, q_low=0.01, q_high=0.99):
    x = np.asarray(x, dtype=float)
    lo = np.nanquantile(x, q_low)
    hi = np.nanquantile(x, q_high)
    return np.clip(x, lo, hi)


def zscore_array(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return np.full(len(x), np.nan)
    return (x - mu) / sd


def corr_np(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3:
        return np.nan
    x = x - x.mean()
    y = y - y.mean()
    den = np.sqrt(np.sum(x ** 2) * np.sum(y ** 2))
    if den <= 0 or not np.isfinite(den):
        return np.nan
    return np.sum(x * y) / den


def rank_corr_np(x, y):
    xr = pd.Series(x).rank(method="average").values
    yr = pd.Series(y).rank(method="average").values
    return corr_np(xr, yr)


def calc_ir(s):
    s = pd.Series(s).dropna()
    sd = s.std(ddof=1)
    if len(s) < 2 or sd == 0 or np.isnan(sd):
        return np.nan
    return s.mean() / sd


# =========================
# 3. 获取调仓日期
# =========================

progress("开始获取交易日列表")
trade_dates_sql = f"""
SELECT DISTINCT date
FROM cn_stock_bar1d
WHERE date >= '{START_DATE}'
  AND date <= '{END_DATE}'
ORDER BY date
"""
trade_dates_df = query_df(trade_dates_sql)
trade_dates_df["date"] = pd.to_datetime(trade_dates_df["date"])
trade_dates = trade_dates_df["date"].drop_duplicates().sort_values().reset_index(drop=True)

if len(trade_dates) == 0:
    raise ValueError("指定时间段内没有交易日数据。")

rebalance_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()
rebalance_date_sql = date_in_sql(rebalance_dates)
progress(f"交易日数量：{len(trade_dates):,}；调仓截面数量：{len(rebalance_dates):,}")

del trade_dates_df
gc.collect()


# =========================
# 4. 计算因子与未来30日收益
# =========================

progress("开始计算 hml_r_std_5m 与未来30日收益")

factor_df = None
if USE_SQL_FACTOR_CALC:
    sql_factor_candidates = [
        f"""
        WITH base AS (
            SELECT
                date,
                instrument,
                high,
                low,
                close,
                pre_close
            FROM cn_stock_bar1d
            WHERE date >= '{QUERY_START_DATE}'
              AND date <= '{END_DATE}'
              AND high > 0
              AND low > 0
              AND close > 0
        ), ret AS (
            SELECT
                date,
                instrument,
                close,
                high / pre_close - 1 AS high_r,
                low / pre_close - 1 AS low_r
            FROM base
            WHERE pre_close > 0
        ), fac AS (
            SELECT
                date,
                instrument,
                stddev_samp(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS high_r_std_5m,
                stddev_samp(low_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS low_r_std_5m,
                count(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS obs_cnt,
                lead(close, {REBALANCE_DAYS}) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                ) / close - 1 AS future_ret_30d
            FROM ret
        )
        SELECT
            date,
            instrument,
            high_r_std_5m - low_r_std_5m AS {FACTOR_NAME},
            future_ret_30d
        FROM fac
        WHERE date IN ({rebalance_date_sql})
          AND obs_cnt >= {MIN_OBS}
          AND future_ret_30d IS NOT NULL
        """,
        f"""
        WITH base AS (
            SELECT
                date,
                instrument,
                high,
                low,
                close,
                lag(close) OVER (PARTITION BY instrument ORDER BY date) AS pre_close
            FROM cn_stock_bar1d
            WHERE date >= '{QUERY_START_DATE}'
              AND date <= '{END_DATE}'
              AND high > 0
              AND low > 0
              AND close > 0
        ), ret AS (
            SELECT
                date,
                instrument,
                close,
                high / pre_close - 1 AS high_r,
                low / pre_close - 1 AS low_r
            FROM base
            WHERE pre_close > 0
        ), fac AS (
            SELECT
                date,
                instrument,
                stddev_samp(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS high_r_std_5m,
                stddev_samp(low_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS low_r_std_5m,
                count(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS obs_cnt,
                lead(close, {REBALANCE_DAYS}) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                ) / close - 1 AS future_ret_30d
            FROM ret
        )
        SELECT
            date,
            instrument,
            high_r_std_5m - low_r_std_5m AS {FACTOR_NAME},
            future_ret_30d
        FROM fac
        WHERE date IN ({rebalance_date_sql})
          AND obs_cnt >= {MIN_OBS}
          AND future_ret_30d IS NOT NULL
        """,
    ]
    try:
        factor_df = first_success_query(
            sql_factor_candidates,
            "SQL侧计算 hml_r_std_5m 失败，可能是表字段或窗口函数不兼容"
        )
    except Exception as e:
        progress(f"SQL侧计算失败，切换为 Python 侧滚动计算。原因：{e}")
        factor_df = None

if factor_df is None:
    progress("开始 Python 侧滚动计算")
    bar_sql_candidates = [
        f"""
        SELECT date, instrument, high, low, close, pre_close
        FROM cn_stock_bar1d
        WHERE date >= '{QUERY_START_DATE}'
          AND date <= '{END_DATE}'
        ORDER BY instrument, date
        """,
        f"""
        SELECT date, instrument, high, low, close
        FROM cn_stock_bar1d
        WHERE date >= '{QUERY_START_DATE}'
          AND date <= '{END_DATE}'
        ORDER BY instrument, date
        """,
    ]
    bar = first_success_query(
        bar_sql_candidates,
        "无法从 cn_stock_bar1d 读取计算因子所需字段"
    )
    bar["date"] = pd.to_datetime(bar["date"])
    bar = downcast_float(bar, ["high", "low", "close", "pre_close"])
    bar = bar.dropna(subset=["date", "instrument", "high", "low", "close"])
    bar = bar[(bar["high"] > 0) & (bar["low"] > 0) & (bar["close"] > 0)].copy()
    bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)
    progress_rows("Python侧日行情数据", bar)

    if "pre_close" not in bar.columns or bar["pre_close"].isna().all():
        bar["pre_close"] = bar.groupby("instrument", sort=False)["close"].shift(1)

    bar["high_r"] = bar["high"] / bar["pre_close"] - 1
    bar["low_r"] = bar["low"] / bar["pre_close"] - 1
    bar.loc[~np.isfinite(bar["high_r"]), "high_r"] = np.nan
    bar.loc[~np.isfinite(bar["low_r"]), "low_r"] = np.nan

    bar["high_r_std_5m"] = (
        bar.groupby("instrument", sort=False)["high_r"]
        .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
        .std()
        .reset_index(level=0, drop=True)
    )
    bar["low_r_std_5m"] = (
        bar.groupby("instrument", sort=False)["low_r"]
        .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
        .std()
        .reset_index(level=0, drop=True)
    )
    bar[FACTOR_NAME] = bar["high_r_std_5m"] - bar["low_r_std_5m"]
    bar["future_close_30d"] = bar.groupby("instrument", sort=False)["close"].shift(-REBALANCE_DAYS)
    bar["future_ret_30d"] = bar["future_close_30d"] / bar["close"] - 1

    factor_df = bar.loc[
        bar["date"].isin(rebalance_dates),
        ["date", "instrument", FACTOR_NAME, "future_ret_30d"]
    ].copy()
    del bar
    gc.collect()

factor_df["date"] = pd.to_datetime(factor_df["date"])
factor_df = downcast_float(factor_df, [FACTOR_NAME, "future_ret_30d"])
factor_df = factor_df.dropna(subset=["date", "instrument", FACTOR_NAME, "future_ret_30d"])
factor_df = factor_df[np.isfinite(factor_df[FACTOR_NAME]) & np.isfinite(factor_df["future_ret_30d"])].copy()
factor_df = factor_df.drop_duplicates(subset=["date", "instrument"], keep="last")
progress_rows("调仓截面因子数据", factor_df)


# =========================
# 5. 读取调仓截面市值数据
# =========================

progress("开始读取调仓截面市值数据")

mkt_table_candidates = [
    "cn_stock_valuation",
    "cn_stock_prefactors",
    "cn_stock_factors",
    "cn_stock_derivative_indicator",
    "cn_stock_bar1d",
]
mkt_date_candidates = ["date", "trade_date"]
mkt_instrument_candidates = ["instrument", "order_book_id", "ts_code", "symbol"]
mkt_col_candidates = [
    "total_market_cap",
    "market_cap",
    "total_mv",
    "mkt_cap",
    "total_market_value",
    "circ_market_cap",
    "float_market_cap",
    "circ_mv",
    "float_mv",
]

mkt = None
mkt_source = None
for table in mkt_table_candidates:
    for date_col in mkt_date_candidates:
        for instrument_col in mkt_instrument_candidates:
            for mkt_col in mkt_col_candidates:
                sql = f"""
                SELECT
                    {date_col} AS date,
                    {instrument_col} AS instrument,
                    {mkt_col} AS mkt_cap
                FROM {table}
                WHERE {date_col} IN ({rebalance_date_sql})
                """
                tmp = try_query(sql)
                if tmp is not None:
                    mkt = tmp.copy()
                    mkt_source = f"{table}.{mkt_col}"
                    break
            if mkt is not None:
                break
        if mkt is not None:
            break
    if mkt is not None:
        break

if mkt is None or len(mkt) == 0:
    raise ValueError("未能读取市值字段。请确认平台中可用的市值表或字段名称。")

mkt["date"] = pd.to_datetime(mkt["date"])
mkt["mkt_cap"] = pd.to_numeric(mkt["mkt_cap"], errors="coerce")
mkt = mkt.dropna(subset=["date", "instrument", "mkt_cap"])
mkt = mkt[mkt["mkt_cap"] > 0].copy()
mkt = mkt.drop_duplicates(subset=["date", "instrument"], keep="last")
mkt = downcast_float(mkt, ["mkt_cap"])
progress_rows(f"市值数据来源 {mkt_source}", mkt)


# =========================
# 6. 读取调仓截面行业数据
# =========================

progress("开始读取行业数据")

industry_table_candidates = [
    "cn_stock_industry_component",
    "cn_stock_industry",
    "cn_stock_industry_classification",
    "cn_stock_basic_info",
    "cn_stock_prefactors",
    "cn_stock_factors",
]
industry_date_candidates = ["date", "trade_date"]
industry_instrument_candidates = ["instrument", "order_book_id", "ts_code", "symbol"]
industry_col_candidates = [
    "sw2021_level1",
    "sw_level1",
    "sw_l1",
    "industry_level1",
    "industry_name_level1",
    "industry_l1",
    "industry_name",
    "industry",
    "sector",
]

industry = None
industry_source = None
industry_has_date = False

for table in industry_table_candidates:
    for date_col in industry_date_candidates:
        for instrument_col in industry_instrument_candidates:
            for industry_col in industry_col_candidates:
                sql = f"""
                SELECT
                    {date_col} AS date,
                    {instrument_col} AS instrument,
                    {industry_col} AS industry
                FROM {table}
                WHERE {date_col} IN ({rebalance_date_sql})
                """
                tmp = try_query(sql)
                if tmp is not None:
                    industry = tmp.copy()
                    industry_source = f"{table}.{industry_col}"
                    industry_has_date = True
                    break
            if industry is not None:
                break
        if industry is not None:
            break
    if industry is not None:
        break

if industry is None:
    for table in industry_table_candidates:
        for instrument_col in industry_instrument_candidates:
            for industry_col in industry_col_candidates:
                sql = f"""
                SELECT
                    {instrument_col} AS instrument,
                    {industry_col} AS industry
                FROM {table}
                """
                tmp = try_query(sql)
                if tmp is not None:
                    industry = tmp.copy()
                    industry_source = f"{table}.{industry_col}"
                    industry_has_date = False
                    break
            if industry is not None:
                break
        if industry is not None:
            break

if industry is None or len(industry) == 0:
    raise ValueError("未能读取行业字段。请确认平台中可用的行业表或字段名称。")

industry["industry"] = industry["industry"].astype(str)
industry = industry[industry["industry"].notna() & (industry["industry"] != "nan")].copy()

if industry_has_date:
    industry["date"] = pd.to_datetime(industry["date"])
    industry = industry.drop_duplicates(subset=["date", "instrument"], keep="last")
else:
    industry = industry.drop_duplicates(subset=["instrument"], keep="last")

progress_rows(f"行业数据来源 {industry_source}", industry)


# =========================
# 7. 合并调仓截面数据
# =========================

progress("开始合并调仓截面因子、市值、行业数据")

test_df = factor_df.merge(mkt, on=["date", "instrument"], how="inner")
del factor_df, mkt
gc.collect()

if industry_has_date:
    test_df = test_df.merge(industry[["date", "instrument", "industry"]], on=["date", "instrument"], how="inner")
else:
    test_df = test_df.merge(industry[["instrument", "industry"]], on="instrument", how="inner")
del industry
gc.collect()

test_df = test_df.dropna(subset=[FACTOR_NAME, "future_ret_30d", "mkt_cap", "industry"])
test_df = test_df[(test_df["mkt_cap"] > 0) & np.isfinite(test_df["future_ret_30d"])].copy()
test_df["log_mkt_cap"] = np.log(test_df["mkt_cap"].astype(float))
test_df["industry"] = test_df["industry"].astype("category")
progress_rows("合并后调仓截面数据", test_df)

if len(test_df) == 0:
    raise ValueError("合并后没有有效样本，请检查市值或行业数据是否与行情代码、日期匹配。")


# =========================
# 8. 截面去极值、市值行业中性化、市值分组
# =========================
# 说明：
# 原始写法通常会对每个截面构造行业哑变量矩阵，再做 np.linalg.lstsq。
# 在行业字段维度异常偏高，或者平台返回的 industry_name 不是一级行业时，
# 哑变量矩阵会很宽，计算会明显变慢甚至看起来“卡住”。
# 这里使用 Frisch-Waugh-Lovell 等价变换：
#   factor ~ log_mkt_cap + industry dummies
# 等价于：
#   先分别对 factor 和 log_mkt_cap 做行业内去均值，
#   再用去均值后的 factor 对去均值后的 log_mkt_cap 做一元回归，
#   残差即为市值行业中性化后的因子。
# 该方法不改变信息集，和显式行业哑变量回归在数学上等价，但更快、更省内存。

progress("开始逐截面去极值、市值行业中性化与15档市值分组")

all_dates = sorted(test_df["date"].drop_duplicates())
parts = []

unique_industry_count = test_df["industry"].astype(str).nunique(dropna=True)
progress(f"行业字段去重数量：{unique_industry_count:,}")

if unique_industry_count > 120:
    progress("警告：行业字段去重数量偏高，可能读取到的不是一级行业字段；代码仍会继续，但建议确认行业字段。")


def neutralize_one_cross_section_fast(g):
    g = g[["date", "instrument", FACTOR_NAME, "future_ret_30d", "mkt_cap", "log_mkt_cap", "industry"]].copy()

    if len(g) < max(100, N_SIZE_GROUPS * MIN_CROSS_SECTION // 2):
        return None

    g[FACTOR_NAME] = pd.to_numeric(g[FACTOR_NAME], errors="coerce")
    g["future_ret_30d"] = pd.to_numeric(g["future_ret_30d"], errors="coerce")
    g["mkt_cap"] = pd.to_numeric(g["mkt_cap"], errors="coerce")
    g["log_mkt_cap"] = pd.to_numeric(g["log_mkt_cap"], errors="coerce")
    g["industry"] = g["industry"].astype(str)

    g = g.dropna(subset=[FACTOR_NAME, "future_ret_30d", "mkt_cap", "log_mkt_cap", "industry"])
    g = g[
        np.isfinite(g[FACTOR_NAME].values) &
        np.isfinite(g["future_ret_30d"].values) &
        np.isfinite(g["mkt_cap"].values) &
        np.isfinite(g["log_mkt_cap"].values) &
        (g["mkt_cap"].values > 0)
    ].copy()

    if len(g) < max(100, N_SIZE_GROUPS * MIN_CROSS_SECTION // 2):
        return None

    # 截面去极值
    g["factor_w"] = winsorize_array(g[FACTOR_NAME].values, WINSOR_Q_LOW, WINSOR_Q_HIGH)

    # 市值先做截面标准化，避免数值尺度影响
    g["log_mkt_z"] = zscore_array(g["log_mkt_cap"].values)
    g = g.dropna(subset=["factor_w", "log_mkt_z"])

    if len(g) < max(100, N_SIZE_GROUPS * MIN_CROSS_SECTION // 2):
        return None

    # 行业固定效应残差化：行业内去均值
    ind_group = g.groupby("industry", sort=False, observed=True)
    y_dm = g["factor_w"] - ind_group["factor_w"].transform("mean")
    x_dm = g["log_mkt_z"] - ind_group["log_mkt_z"].transform("mean")

    y = y_dm.astype(float).values
    x = x_dm.astype(float).values
    valid = np.isfinite(y) & np.isfinite(x)

    if valid.sum() < 100:
        return None

    y_valid = y[valid]
    x_valid = x[valid]
    x_var = np.sum(x_valid ** 2)

    if x_var <= 0 or not np.isfinite(x_var):
        return None

    beta = np.sum(x_valid * y_valid) / x_var
    resid_valid = y_valid - beta * x_valid

    out = g.loc[valid, ["date", "future_ret_30d", "mkt_cap"]].copy()
    out["factor_resid"] = resid_valid
    out["factor_z"] = zscore_array(out["factor_resid"].values)

    # 每个截面内按市值升序分15组：1=小市值，15=大市值
    rank = out["mkt_cap"].rank(method="first", ascending=True)
    try:
        out["市值分组"] = pd.qcut(
            rank,
            q=N_SIZE_GROUPS,
            labels=list(range(1, N_SIZE_GROUPS + 1))
        ).astype(int)
    except Exception:
        return None

    out = out.dropna(subset=["factor_z", "future_ret_30d", "市值分组"])
    if len(out) == 0:
        return None

    return out[["date", "市值分组", "factor_z", "future_ret_30d"]]


for i, dt in enumerate(all_dates, 1):
    g = test_df.loc[test_df["date"] == dt]

    if i == 1 or i % 5 == 0 or i == len(all_dates):
        progress(f"中性化进度：{i}/{len(all_dates)}，当前截面 {pd.to_datetime(dt).strftime('%Y-%m-%d')}，样本 {len(g):,}")

    out = neutralize_one_cross_section_fast(g)
    if out is not None and len(out) > 0:
        parts.append(out)

if len(parts) == 0:
    raise ValueError("没有形成有效的中性化截面样本。")

neutral_df = pd.concat(parts, ignore_index=True)
del parts, test_df
gc.collect()
progress_rows("中性化并分组后的样本", neutral_df)


# =========================
# 9. 计算每期、每组指标
# =========================

progress("开始计算每期、每个市值组的 IC、RankIC、因子收益率、t值")

records = []
grouped = neutral_df.groupby(["date", "市值分组"], sort=True)
total_groups = grouped.ngroups

for j, ((dt, size_group), sub) in enumerate(grouped, 1):
    n = len(sub)
    if n < MIN_CROSS_SECTION:
        continue

    x = sub["factor_z"].astype(float).values
    y = sub["future_ret_30d"].astype(float).values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    n = len(x)
    if n < MIN_CROSS_SECTION:
        continue

    x_mean = x.mean()
    y_mean = y.mean()
    x_demean = x - x_mean
    y_demean = y - y_mean
    x_var = np.sum(x_demean ** 2)

    if x_var <= 0 or not np.isfinite(x_var):
        continue

    ic = corr_np(x, y)
    rank_ic = rank_corr_np(x, y)
    beta = np.sum(x_demean * y_demean) / x_var

    residual = y - (y_mean + beta * x_demean)
    dof = n - 2
    if dof > 0:
        sigma2 = np.sum(residual ** 2) / dof
        se_beta = np.sqrt(sigma2 / x_var)
        t_value = beta / se_beta if se_beta > 0 else np.nan
    else:
        t_value = np.nan

    records.append({
        "截面日期": dt,
        "市值分组": int(size_group),
        "样本数": int(n),
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": beta,
        "t值": t_value,
    })

    if j % 200 == 0 or j == total_groups:
        progress(f"指标计算进度：{j}/{total_groups}")

if len(records) == 0:
    raise ValueError("没有得到有效的市值分组指标，请检查 MIN_CROSS_SECTION 是否过高。")

ts_metrics = pd.DataFrame(records)
del records, neutral_df
gc.collect()
progress_rows("分期分组指标", ts_metrics)


# =========================
# 10. 汇总15组指标并输出
# =========================

progress("开始汇总15档市值分组结果")

summary = (
    ts_metrics.groupby("市值分组")
    .agg(
        有效截面数=("截面日期", "count"),
        平均样本数=("样本数", "mean"),
        IC均值=("IC", "mean"),
        ICIR=("IC", calc_ir),
        RankIC均值=("RankIC", "mean"),
        RankICIR=("RankIC", calc_ir),
        因子收益率均值=("因子收益率", "mean"),
        t值均值=("t值", "mean"),
    )
    .reset_index()
    .sort_values("市值分组")
)

summary.insert(0, "因子", FACTOR_NAME + "_市值行业中性化")
summary.insert(
    2,
    "分组说明",
    summary["市值分组"].map(lambda x: "小市值" if x == 1 else ("大市值" if x == N_SIZE_GROUPS else ""))
)

progress("完成")
display(summary.round(6))


可以看出该因子在不同市值区间内表现会有所差异，但目前仍不能只凭借指标就判断该因子在组合策略当中应该真正适用哪些市值区间，于是考虑构建一版可选市值区间回测策略，并且剔除掉研报当中明确体现的表现不佳的行业

## 可选市值区间回测策略

In [ ]:
# ============================================================
# hml_r_std_5m 市值行业中性化因子策略回测
# 逻辑：全市场按市值分15组，在指定市值组内分别选因子值最低5%的股票，等权持有
# 调仓周期：30个交易日
# 交易约束：信号日剔除指定行业、可识别的ST、*ST、退市股票；停牌和涨跌停由BigTrader成交撮合处理；考虑交易成本
# 严谨性：不在信号日读取下一交易日行情、涨跌停、成交量、停牌等信息，避免潜在未来函数
# 稳定性：固定行业剔除、去重、排序、分组、选股数量规则，降低多次回测的微小差异
# 回测引擎：BigQuant BigTrader 原生回测引擎
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import gc
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e

try:
    from bigquant import bigtrader
except Exception:
    try:
        import bigtrader
    except Exception as e:
        raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 BigTrader。") from e


# =========================
# 1. 参数设置
# =========================

START_DATE = "2022-01-01"
END_DATE = "2026-06-30"

FACTOR_NAME = "hml_r_std_5m"
NEUTRAL_FACTOR_NAME = "factor_mkt_ind_neutral"

# 人工指定参与选股的市值组；1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [1, 2, 3]

N_SIZE_GROUPS = 15
BOTTOM_PCT = 0.3
REBALANCE_DAYS = 30
LOOKBACK_DAYS = 105
MIN_OBS = 80
MIN_STOCKS_PER_SIZE_GROUP = 20

# 行业剔除列表：根据用户图片，保留“非银行金融”，剔除其余列示行业。
# 注意：这里先剔除指定行业，再在剩余股票池内进行市值行业中性化、市值15组分组和选股。
EXCLUDED_INDUSTRIES = [
    "商贸零售",
    "计算机",
    "医药",
    "轻工制造",
    "食品饮料",
    "农林牧渔",
    "传媒",
    "建材",
    "综合",
    "交通运输",
    "通信",
    "银行",
]

# 防止“银行”误伤“非银行金融/非银金融”。
KEEP_INDUSTRIES = ["非银行金融", "非银行业金融", "非银金融"]

# 因子最低5%的取整规则：floor表示向下取整，至少取1只；该规则固定，避免每次理解不一致。
SELECT_COUNT_METHOD = "floor"

WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99

CAPITAL_BASE = 1_000_000
BENCHMARK = "932000.CSI"

# 交易成本：买入佣金 + 卖出佣金与印花税。可按需要调整。
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# 是否优先尝试在SQL侧计算滚动因子，失败后自动切换Python侧计算
USE_SQL_FACTOR_CALC = True

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")
QUERY_START_DATE = (pd.to_datetime(START_DATE) - pd.Timedelta(days=260)).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")

bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if int(g) < 1 or int(g) > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")

SIZE_GROUPS_TO_TRADE = sorted(set([int(x) for x in SIZE_GROUPS_TO_TRADE]))
EXCLUDED_INDUSTRIES = list(dict.fromkeys([str(x).strip() for x in EXCLUDED_INDUSTRIES if str(x).strip()]))
KEEP_INDUSTRIES = list(dict.fromkeys([str(x).strip() for x in KEEP_INDUSTRIES if str(x).strip()]))

if SELECT_COUNT_METHOD not in ["floor", "ceil"]:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


# =========================
# 2. 工具函数
# =========================

_T0 = time.time()


def _elapsed():
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg):
    print(f"[{_elapsed()}] {msg}", flush=True)


def mem_mb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1024 / 1024
    except Exception:
        return np.nan


def progress_rows(name, df):
    m = mem_mb()
    if np.isfinite(m):
        progress(f"{name}：{len(df):,} 行，内存约 {m:.1f} MB")
    else:
        progress(f"{name}：{len(df):,} 行")


def query_df(sql):
    return dai.query(sql).df()


def try_query(sql):
    try:
        df = query_df(sql)
        if df is not None and len(df) > 0:
            return df
    except Exception:
        return None
    return None


def first_success_query(sql_list, err_msg):
    last_error = None
    for sql in sql_list:
        try:
            df = query_df(sql)
            if df is not None and len(df) > 0:
                return df
        except Exception as e:
            last_error = e
            continue
    raise ValueError(f"{err_msg}。最后一次错误：{last_error}")


def date_in_sql(dates):
    return ", ".join([f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates])


def downcast_float(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def winsorize_array(x, q_low=0.01, q_high=0.99):
    x = np.asarray(x, dtype=float)
    lo = np.nanquantile(x, q_low)
    hi = np.nanquantile(x, q_high)
    return np.clip(x, lo, hi)


def zscore_array(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return np.full(len(x), np.nan)
    return (x - mu) / sd


def to_date_str(x):
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def normalize_industry_name(x):
    """将不同来源的行业名称做轻量归一化，便于稳定剔除。"""
    if pd.isna(x):
        return ""
    s = str(x).strip()
    for token in [
        "申万一级行业", "申万一级", "申万", "SW2021", "SW", "中信一级行业", "中信一级",
        "一级行业", "行业", "（", "）", "(", ")", " ", "-", "_"
    ]:
        s = s.replace(token, "")
    alias = {
        "非银金融": "非银行金融",
        "非银行业金融": "非银行金融",
        "商业贸易": "商贸零售",
        "商贸零售业": "商贸零售",
        "食品饮料业": "食品饮料",
        "农林牧渔业": "农林牧渔",
        "轻工制造业": "轻工制造",
        "交通运输业": "交通运输",
        "建筑材料": "建材",
        "银行业": "银行",
        "通信设备": "通信",
        "医药生物": "医药",
    }
    return alias.get(s, s)


EXCLUDED_INDUSTRIES_NORM = set(normalize_industry_name(x) for x in EXCLUDED_INDUSTRIES)
KEEP_INDUSTRIES_NORM = set(normalize_industry_name(x) for x in KEEP_INDUSTRIES)


def is_excluded_industry(x):
    """判断行业是否在剔除名单内；非银行金融显式保留，避免被“银行”误伤。"""
    s = normalize_industry_name(x)
    if not s:
        return False
    if s in KEEP_INDUSTRIES_NORM:
        return False
    if "非银行金融" in s or "非银金融" in s or "非银行业金融" in s:
        return False
    if s in EXCLUDED_INDUSTRIES_NORM:
        return True
    # 兼容“xx银行”等名称，但仍不匹配非银行金融。
    if "银行" in s and "非银行" not in s and "非银" not in s:
        return True
    for ex in EXCLUDED_INDUSTRIES_NORM:
        if ex == "银行":
            continue
        if ex and (ex in s or s in ex):
            return True
    return False


def calc_select_count(n, pct):
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def get_current_date_from_engine(context, data):
    if data is not None and hasattr(data, "current_dt"):
        return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            v = getattr(context, attr)
            if v is not None:
                return pd.to_datetime(v).strftime("%Y-%m-%d")
    return None


def get_positions_dict(context):
    for method in ["get_account_positions", "get_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    return {}


def position_amount(pos_obj):
    try:
        return float(getattr(pos_obj, "amount", 0))
    except Exception:
        try:
            return float(pos_obj.get("amount", 0))
        except Exception:
            return 0.0


def order_to_target_percent(context, instrument, weight):
    try:
        context.order_target_percent(instrument, float(weight))
        return True
    except Exception:
        try:
            # 兼容部分文档中的 order_percent 写法；在BigTrader股票策略中通常也表示目标仓位比例。
            context.order_percent(instrument, float(weight))
            return True
        except Exception as e:
            print(f"下单失败：{instrument}, target={weight:.6f}, err={e}", flush=True)
            return False


# =========================
# 3. 获取交易日和调仓信号日期
# =========================

progress("开始获取交易日列表")
trade_dates_sql = f"""
SELECT DISTINCT date
FROM cn_stock_bar1d
WHERE date >= '{START_DATE}'
  AND date <= '{END_DATE}'
ORDER BY date
"""
trade_dates_df = query_df(trade_dates_sql)
trade_dates_df["date"] = pd.to_datetime(trade_dates_df["date"])
trade_dates = trade_dates_df["date"].drop_duplicates().sort_values().reset_index(drop=True)

if len(trade_dates) < REBALANCE_DAYS + 2:
    raise ValueError("指定时间段内交易日过少，无法完成30日调仓回测。")

signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()

# BigTrader日线回测通常在当前K线提交订单，并在下一根K线撮合成交；
# 因此这里用 signal_date 计算因子和组合，订单实际在 next_trade_date 交易。
signal_to_execution = {}
for dt in signal_dates:
    idx = trade_dates[trade_dates == dt].index
    if len(idx) == 0:
        continue
    next_idx = int(idx[0]) + 1
    if next_idx < len(trade_dates):
        signal_to_execution[to_date_str(dt)] = to_date_str(trade_dates.iloc[next_idx])

signal_dates = [pd.to_datetime(k) for k in signal_to_execution.keys()]
if len(signal_dates) == 0:
    raise ValueError("没有可用调仓信号日期。")

signal_date_sql = date_in_sql(signal_dates)
progress(f"交易日数量：{len(trade_dates):,}；调仓信号截面数量：{len(signal_dates):,}")
progress(f"回测使用市值组：{SIZE_GROUPS_TO_TRADE}；每组选取最低 {BOTTOM_PCT:.1%}；选股数量取整：{SELECT_COUNT_METHOD}")
progress(f"剔除行业：{EXCLUDED_INDUSTRIES}；显式保留行业：{KEEP_INDUSTRIES}")

del trade_dates_df
gc.collect()


# =========================
# 4. 计算 hml_r_std_5m 因子
# =========================

progress("开始计算 hml_r_std_5m 因子")

factor_df = None
if USE_SQL_FACTOR_CALC:
    sql_factor_candidates = [
        f"""
        WITH base AS (
            SELECT
                date,
                instrument,
                high,
                low,
                close,
                pre_close
            FROM cn_stock_bar1d
            WHERE date >= '{QUERY_START_DATE}'
              AND date <= '{END_DATE}'
              AND high > 0
              AND low > 0
              AND close > 0
        ), ret AS (
            SELECT
                date,
                instrument,
                high / pre_close - 1 AS high_r,
                low / pre_close - 1 AS low_r
            FROM base
            WHERE pre_close > 0
        ), fac AS (
            SELECT
                date,
                instrument,
                stddev_samp(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS high_r_std_5m,
                stddev_samp(low_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS low_r_std_5m,
                count(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS obs_cnt
            FROM ret
        )
        SELECT
            date,
            instrument,
            high_r_std_5m - low_r_std_5m AS {FACTOR_NAME}
        FROM fac
        WHERE date IN ({signal_date_sql})
          AND obs_cnt >= {MIN_OBS}
        """,
        f"""
        WITH base AS (
            SELECT
                date,
                instrument,
                high,
                low,
                close,
                lag(close) OVER (PARTITION BY instrument ORDER BY date) AS pre_close
            FROM cn_stock_bar1d
            WHERE date >= '{QUERY_START_DATE}'
              AND date <= '{END_DATE}'
              AND high > 0
              AND low > 0
              AND close > 0
        ), ret AS (
            SELECT
                date,
                instrument,
                high / pre_close - 1 AS high_r,
                low / pre_close - 1 AS low_r
            FROM base
            WHERE pre_close > 0
        ), fac AS (
            SELECT
                date,
                instrument,
                stddev_samp(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS high_r_std_5m,
                stddev_samp(low_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS low_r_std_5m,
                count(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS obs_cnt
            FROM ret
        )
        SELECT
            date,
            instrument,
            high_r_std_5m - low_r_std_5m AS {FACTOR_NAME}
        FROM fac
        WHERE date IN ({signal_date_sql})
          AND obs_cnt >= {MIN_OBS}
        """,
    ]
    try:
        factor_df = first_success_query(
            sql_factor_candidates,
            "SQL侧计算 hml_r_std_5m 失败，可能是表字段或窗口函数不兼容"
        )
    except Exception as e:
        progress(f"SQL侧计算失败，切换为 Python 侧滚动计算。原因：{e}")
        factor_df = None

if factor_df is None:
    progress("开始 Python 侧滚动计算")
    bar_sql_candidates = [
        f"""
        SELECT date, instrument, high, low, close, pre_close
        FROM cn_stock_bar1d
        WHERE date >= '{QUERY_START_DATE}'
          AND date <= '{END_DATE}'
        ORDER BY instrument, date
        """,
        f"""
        SELECT date, instrument, high, low, close
        FROM cn_stock_bar1d
        WHERE date >= '{QUERY_START_DATE}'
          AND date <= '{END_DATE}'
        ORDER BY instrument, date
        """,
    ]
    bar = first_success_query(bar_sql_candidates, "无法从 cn_stock_bar1d 读取计算因子所需字段")
    bar["date"] = pd.to_datetime(bar["date"])
    bar = downcast_float(bar, ["high", "low", "close", "pre_close"])
    bar = bar.dropna(subset=["date", "instrument", "high", "low", "close"])
    bar = bar[(bar["high"] > 0) & (bar["low"] > 0) & (bar["close"] > 0)].copy()
    bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)
    progress_rows("Python侧日行情数据", bar)

    if "pre_close" not in bar.columns or bar["pre_close"].isna().all():
        bar["pre_close"] = bar.groupby("instrument", sort=False)["close"].shift(1)

    bar["high_r"] = bar["high"] / bar["pre_close"] - 1
    bar["low_r"] = bar["low"] / bar["pre_close"] - 1
    bar.loc[~np.isfinite(bar["high_r"]), "high_r"] = np.nan
    bar.loc[~np.isfinite(bar["low_r"]), "low_r"] = np.nan

    bar["high_r_std_5m"] = (
        bar.groupby("instrument", sort=False)["high_r"]
        .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
        .std()
        .reset_index(level=0, drop=True)
    )
    bar["low_r_std_5m"] = (
        bar.groupby("instrument", sort=False)["low_r"]
        .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
        .std()
        .reset_index(level=0, drop=True)
    )
    bar[FACTOR_NAME] = bar["high_r_std_5m"] - bar["low_r_std_5m"]
    factor_df = bar.loc[bar["date"].isin(signal_dates), ["date", "instrument", FACTOR_NAME]].copy()
    del bar
    gc.collect()

factor_df["date"] = pd.to_datetime(factor_df["date"])
factor_df = downcast_float(factor_df, [FACTOR_NAME])
factor_df = factor_df.dropna(subset=["date", "instrument", FACTOR_NAME])
factor_df = factor_df[np.isfinite(factor_df[FACTOR_NAME])].copy()
factor_df = factor_df.sort_values(["date", "instrument"], kind="mergesort")
factor_df = factor_df.drop_duplicates(subset=["date", "instrument"], keep="last")
progress_rows("调仓截面因子数据", factor_df)


# =========================
# 5. 读取调仓截面市值数据
# =========================

progress("开始读取调仓截面市值数据")

mkt_table_candidates = [
    "cn_stock_valuation",
    "cn_stock_prefactors",
    "cn_stock_factors",
    "cn_stock_derivative_indicator",
    "cn_stock_bar1d",
]
mkt_date_candidates = ["date", "trade_date"]
mkt_instrument_candidates = ["instrument", "order_book_id", "ts_code", "symbol"]
mkt_col_candidates = [
    "total_market_cap",
    "market_cap",
    "total_mv",
    "mkt_cap",
    "total_market_value",
    "circ_market_cap",
    "float_market_cap",
    "circ_mv",
    "float_mv",
]

mkt = None
mkt_source = None
for table in mkt_table_candidates:
    for date_col in mkt_date_candidates:
        for instrument_col in mkt_instrument_candidates:
            for mkt_col in mkt_col_candidates:
                sql = f"""
                SELECT
                    {date_col} AS date,
                    {instrument_col} AS instrument,
                    {mkt_col} AS mkt_cap
                FROM {table}
                WHERE {date_col} IN ({signal_date_sql})
                """
                tmp = try_query(sql)
                if tmp is not None:
                    mkt = tmp.copy()
                    mkt_source = f"{table}.{mkt_col}"
                    break
            if mkt is not None:
                break
        if mkt is not None:
            break
    if mkt is not None:
        break

if mkt is None or len(mkt) == 0:
    raise ValueError("未能读取市值字段。请确认平台中可用的市值表或字段名称。")

mkt["date"] = pd.to_datetime(mkt["date"])
mkt["mkt_cap"] = pd.to_numeric(mkt["mkt_cap"], errors="coerce")
mkt = mkt.dropna(subset=["date", "instrument", "mkt_cap"])
mkt = mkt[mkt["mkt_cap"] > 0].copy()
mkt = mkt.sort_values(["date", "instrument"], kind="mergesort")
mkt = mkt.drop_duplicates(subset=["date", "instrument"], keep="last")
mkt = downcast_float(mkt, ["mkt_cap"])
progress_rows(f"市值数据来源 {mkt_source}", mkt)


# =========================
# 6. 读取调仓截面行业数据
# =========================

progress("开始读取行业数据")

industry_table_candidates = [
    "cn_stock_industry_component",
    "cn_stock_industry",
    "cn_stock_industry_classification",
    "cn_stock_basic_info",
    "cn_stock_prefactors",
    "cn_stock_factors",
]
industry_date_candidates = ["date", "trade_date"]
industry_instrument_candidates = ["instrument", "order_book_id", "ts_code", "symbol"]
industry_col_candidates = [
    "sw2021_level1",
    "sw_level1",
    "sw_l1",
    "industry_level1",
    "industry_name_level1",
    "industry_l1",
    "industry_name",
    "industry",
    "sector",
]

industry = None
industry_source = None
industry_has_date = False

for table in industry_table_candidates:
    for date_col in industry_date_candidates:
        for instrument_col in industry_instrument_candidates:
            for industry_col in industry_col_candidates:
                sql = f"""
                SELECT
                    {date_col} AS date,
                    {instrument_col} AS instrument,
                    {industry_col} AS industry
                FROM {table}
                WHERE {date_col} IN ({signal_date_sql})
                """
                tmp = try_query(sql)
                if tmp is not None:
                    industry = tmp.copy()
                    industry_source = f"{table}.{industry_col}"
                    industry_has_date = True
                    break
            if industry is not None:
                break
        if industry is not None:
            break
    if industry is not None:
        break

if industry is None:
    for table in industry_table_candidates:
        for instrument_col in industry_instrument_candidates:
            for industry_col in industry_col_candidates:
                sql = f"""
                SELECT
                    {instrument_col} AS instrument,
                    {industry_col} AS industry
                FROM {table}
                """
                tmp = try_query(sql)
                if tmp is not None:
                    industry = tmp.copy()
                    industry_source = f"{table}.{industry_col}"
                    industry_has_date = False
                    break
            if industry is not None:
                break
        if industry is not None:
            break

if industry is None or len(industry) == 0:
    raise ValueError("未能读取行业字段。请确认平台中可用的行业表或字段名称。")

industry["industry"] = industry["industry"].astype(str)
industry = industry[industry["industry"].notna() & (industry["industry"] != "nan")].copy()

if industry_has_date:
    industry["date"] = pd.to_datetime(industry["date"])
    industry = industry.sort_values(["date", "instrument", "industry"], kind="mergesort")
    industry = industry.drop_duplicates(subset=["date", "instrument"], keep="last")
else:
    industry = industry.sort_values(["instrument", "industry"], kind="mergesort")
    industry = industry.drop_duplicates(subset=["instrument"], keep="last")

progress_rows(f"行业数据来源 {industry_source}", industry)


# =========================
# 7. 读取信号日 ST、退市状态数据
# =========================
# 严谨性说明：
# - 这里只尝试读取带日期的状态字段，并且只使用 signal_date 当日信息；
# - 不使用不带日期的静态证券名称或当前ST状态做历史过滤，避免把未来状态带入历史回测；
# - 若平台无法提供带日期状态字段，则跳过显式ST过滤，由BigTrader交易数据和撮合规则处理不可交易证券。

progress("开始读取信号日 ST、退市状态数据")

status_table_candidates = [
    "cn_stock_bar1d",
    "cn_stock_status",
    "cn_stock_basic_info",
    "cn_stock_instruments",
    "cn_stock_description",
]
status_date_candidates = ["date", "trade_date"]
status_instrument_candidates = ["instrument", "order_book_id", "ts_code", "symbol"]
name_col_candidates = ["name", "short_name", "sec_name", "security_name", "display_name"]
st_col_candidates = ["is_st", "st", "is_special_treatment", "special_treatment"]
list_status_candidates = ["list_status", "status", "listed_state", "delist_status"]

status = None
status_source = None
status_has_date = False

# 优先：带日期 + 名称 + 可选ST字段
for table in status_table_candidates:
    for date_col in status_date_candidates:
        for instrument_col in status_instrument_candidates:
            for name_col in name_col_candidates:
                for st_col in [None] + st_col_candidates:
                    select_cols = [
                        f"{date_col} AS date",
                        f"{instrument_col} AS instrument",
                        f"{name_col} AS name",
                    ]
                    if st_col is not None:
                        select_cols.append(f"{st_col} AS is_st_flag")
                    sql = f"""
                    SELECT {', '.join(select_cols)}
                    FROM {table}
                    WHERE {date_col} IN ({signal_date_sql})
                    """
                    tmp = try_query(sql)
                    if tmp is not None:
                        status = tmp.copy()
                        status_source = f"{table}.{name_col}" + (f"+{st_col}" if st_col is not None else "")
                        status_has_date = True
                        break
                if status is not None:
                    break
            if status is not None:
                break
        if status is not None:
            break
    if status is not None:
        break

# 备选：带日期 + ST字段，不要求名称字段
if status is None:
    for table in status_table_candidates:
        for date_col in status_date_candidates:
            for instrument_col in status_instrument_candidates:
                for st_col in st_col_candidates:
                    sql = f"""
                    SELECT
                        {date_col} AS date,
                        {instrument_col} AS instrument,
                        {st_col} AS is_st_flag
                    FROM {table}
                    WHERE {date_col} IN ({signal_date_sql})
                    """
                    tmp = try_query(sql)
                    if tmp is not None:
                        status = tmp.copy()
                        status_source = f"{table}.{st_col}"
                        status_has_date = True
                        break
                if status is not None:
                    break
            if status is not None:
                break
        if status is not None:
            break

# 备选：带日期 + 上市状态字段
if status is None:
    for table in status_table_candidates:
        for date_col in status_date_candidates:
            for instrument_col in status_instrument_candidates:
                for list_status_col in list_status_candidates:
                    sql = f"""
                    SELECT
                        {date_col} AS date,
                        {instrument_col} AS instrument,
                        {list_status_col} AS list_status
                    FROM {table}
                    WHERE {date_col} IN ({signal_date_sql})
                    """
                    tmp = try_query(sql)
                    if tmp is not None:
                        status = tmp.copy()
                        status_source = f"{table}.{list_status_col}"
                        status_has_date = True
                        break
                if status is not None:
                    break
            if status is not None:
                break
        if status is not None:
            break

if status is not None and len(status) > 0 and status_has_date:
    status["date"] = pd.to_datetime(status["date"])

    bad = pd.Series(False, index=status.index)

    if "name" in status.columns:
        name_s = status["name"].astype(str)
        bad = bad | name_s.str.contains("ST", case=False, na=False)
        bad = bad | name_s.str.contains("退", na=False)

    if "is_st_flag" in status.columns:
        st_flag = pd.to_numeric(status["is_st_flag"], errors="coerce").fillna(0) != 0
        bad = bad | st_flag

    if "list_status" in status.columns:
        ls = status["list_status"].astype(str).str.lower()
        bad = bad | ls.isin([
            "delisted", "de_listed", "退市", "已退市", "terminated",
            "d", "0", "false", "暂停上市", "终止上市"
        ])

    status["is_bad_status"] = bad.astype(bool)
    status_valid = status[["date", "instrument", "is_bad_status"]].copy()
    status_valid = status_valid.sort_values(["date", "instrument", "is_bad_status"], kind="mergesort")
    status_valid = status_valid.drop_duplicates(subset=["date", "instrument"], keep="last")
    progress_rows(f"状态数据来源 {status_source}", status_valid)
else:
    progress("警告：未读取到可按信号日对齐的ST/退市状态字段；为避免未来函数，不使用静态当前名称或当前状态过滤。")
    status_valid = None
    status_has_date = False


# =========================
# 8. 合并信号截面数据
# =========================

progress("开始合并信号截面数据")

signal_panel = factor_df.merge(mkt, on=["date", "instrument"], how="inner")
del factor_df, mkt
gc.collect()

if industry_has_date:
    signal_panel = signal_panel.merge(industry[["date", "instrument", "industry"]], on=["date", "instrument"], how="inner")
else:
    signal_panel = signal_panel.merge(industry[["instrument", "industry"]], on="instrument", how="inner")
del industry
gc.collect()

signal_panel = signal_panel.dropna(subset=[FACTOR_NAME, "mkt_cap", "industry"])
signal_panel = signal_panel[(signal_panel["mkt_cap"] > 0) & np.isfinite(signal_panel[FACTOR_NAME])].copy()
signal_panel = signal_panel.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)

# 只使用信号日可得的状态字段剔除 ST、*ST、退市股票；该步骤放在中性化和市值分组之前，统一股票池口径。
if status_valid is not None:
    before_n = len(signal_panel)
    signal_panel = signal_panel.merge(
        status_valid[["date", "instrument", "is_bad_status"]],
        on=["date", "instrument"],
        how="left"
    )
    signal_panel["is_bad_status"] = signal_panel["is_bad_status"].fillna(False).astype(bool)
    bad_n = int(signal_panel["is_bad_status"].sum())
    signal_panel = signal_panel[~signal_panel["is_bad_status"]].drop(columns=["is_bad_status"]).copy()
    progress(f"信号日ST/退市剔除：{bad_n:,} 行；剩余 {len(signal_panel):,} 行；剔除前 {before_n:,} 行")
else:
    progress("未使用ST/退市状态过滤：平台未提供可按信号日对齐的状态字段。")

# 剔除指定行业，显式保留非银行金融。
signal_panel["industry_raw"] = signal_panel["industry"].astype(str)
signal_panel["industry_norm"] = signal_panel["industry_raw"].map(normalize_industry_name)
industry_excluded_mask = signal_panel["industry_raw"].map(is_excluded_industry).astype(bool)
excluded_n = int(industry_excluded_mask.sum())
progress(f"行业剔除：{excluded_n:,} 行；剩余 {len(signal_panel) - excluded_n:,} 行")
if excluded_n > 0:
    excluded_dist = (
        signal_panel.loc[industry_excluded_mask, "industry_norm"]
        .value_counts()
        .head(20)
        .reset_index()
    )
    excluded_dist.columns = ["剔除行业", "样本行数"]
    progress("剔除行业样本分布前20：")
    display(excluded_dist)

signal_panel = signal_panel[~industry_excluded_mask].copy()
if len(signal_panel) == 0:
    raise ValueError("行业与状态过滤后没有有效信号样本，请检查剔除行业列表或行业字段。")

signal_panel["log_mkt_cap"] = np.log(signal_panel["mkt_cap"].astype(float))
signal_panel["industry"] = signal_panel["industry_norm"].astype("category")
progress_rows("合并、状态过滤、行业剔除后的信号截面数据", signal_panel)

if len(signal_panel) == 0:
    raise ValueError("合并后没有有效信号样本，请检查行情、市值或行业字段匹配。")


# =========================
# 9. 市值行业中性化、15档市值分组、选股
# =========================
# 严谨性说明：
# - 因子只使用 signal_date 及之前的日频数据；
# - 市值和行业只使用 signal_date 的截面信息；
# - 订单在 signal_date 生成，BigTrader日线机制下通常于下一根K线成交；
# - 不在 signal_date 读取下一交易日行情、成交量、停牌、涨跌停等信息；
# - 涨跌停、停牌导致的无法成交由 BigTrader 撮合机制处理；
# - ST、退市过滤只使用 signal_date 可得的带日期状态字段。

progress("开始逐截面市值行业中性化、15档市值分组与选股")

unique_industry_count = signal_panel["industry"].astype(str).nunique(dropna=True)
progress(f"行业字段去重数量：{unique_industry_count:,}")
if unique_industry_count > 120:
    progress("警告：行业字段去重数量偏高，可能读取到的不是一级行业字段；代码仍会继续。")


def neutralize_group_and_select(g):
    g = g[["date", "instrument", FACTOR_NAME, "mkt_cap", "log_mkt_cap", "industry"]].copy()
    g = g.sort_values(["instrument"], kind="mergesort").reset_index(drop=True)
    if len(g) < max(100, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    g[FACTOR_NAME] = pd.to_numeric(g[FACTOR_NAME], errors="coerce")
    g["mkt_cap"] = pd.to_numeric(g["mkt_cap"], errors="coerce")
    g["log_mkt_cap"] = pd.to_numeric(g["log_mkt_cap"], errors="coerce")
    g["industry"] = g["industry"].astype(str)
    g = g.dropna(subset=[FACTOR_NAME, "mkt_cap", "log_mkt_cap", "industry"])
    g = g[(g["mkt_cap"] > 0) & np.isfinite(g[FACTOR_NAME]) & np.isfinite(g["log_mkt_cap"])].copy()

    if len(g) < max(100, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    # 截面去极值
    g["factor_w"] = winsorize_array(g[FACTOR_NAME].values, WINSOR_Q_LOW, WINSOR_Q_HIGH)

    # 市值标准化后，用FWL方式剔除行业固定效应和市值暴露
    g["log_mkt_z"] = zscore_array(g["log_mkt_cap"].values)
    g = g.dropna(subset=["factor_w", "log_mkt_z"])
    if len(g) < max(100, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    ind_group = g.groupby("industry", sort=True, observed=True)
    y_dm = g["factor_w"] - ind_group["factor_w"].transform("mean")
    x_dm = g["log_mkt_z"] - ind_group["log_mkt_z"].transform("mean")

    y = y_dm.astype(float).values
    x = x_dm.astype(float).values
    valid = np.isfinite(y) & np.isfinite(x)

    if valid.sum() < 100:
        return None

    y_valid = y[valid]
    x_valid = x[valid]
    x_var = np.sum(x_valid ** 2)
    if x_var <= 0 or not np.isfinite(x_var):
        return None

    beta = np.sum(x_valid * y_valid) / x_var
    resid = y_valid - beta * x_valid

    out = g.loc[valid, ["date", "instrument", "mkt_cap"]].copy()
    out[NEUTRAL_FACTOR_NAME] = zscore_array(resid)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "mkt_cap"])

    # 固定排序后再rank，保证市值相同或近似相同时的分组结果稳定。
    out = out.sort_values(["mkt_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = out["mkt_cap"].rank(method="first", ascending=True)
    try:
        out["size_group"] = pd.qcut(
            rank,
            q=N_SIZE_GROUPS,
            labels=list(range(1, N_SIZE_GROUPS + 1))
        ).astype(int)
    except Exception:
        return None

    selected_parts = []
    for size_group in SIZE_GROUPS_TO_TRADE:
        sg = out[out["size_group"] == size_group].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue
        n_select = calc_select_count(len(sg), BOTTOM_PCT)
        sg = sg.sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[True, True], kind="mergesort")
        selected_parts.append(sg.head(n_select))

    if not selected_parts:
        return None

    selected = pd.concat(selected_parts, ignore_index=True)
    return selected[["date", "instrument", "size_group", NEUTRAL_FACTOR_NAME]]


selected_parts = []
all_signal_dates = sorted(signal_panel["date"].drop_duplicates())
for i, dt in enumerate(all_signal_dates, 1):
    g = signal_panel.loc[signal_panel["date"] == dt]
    if i == 1 or i % 5 == 0 or i == len(all_signal_dates):
        progress(f"选股进度：{i}/{len(all_signal_dates)}，信号日 {to_date_str(dt)}，样本 {len(g):,}")
    sel = neutralize_group_and_select(g)
    if sel is not None and len(sel) > 0:
        selected_parts.append(sel)

if not selected_parts:
    raise ValueError("没有形成任何有效选股结果，请检查参数或数据。")

selected_df = pd.concat(selected_parts, ignore_index=True)
del selected_parts, signal_panel
gc.collect()
progress_rows("初步选股结果", selected_df)

selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
selected_df = selected_df.dropna(subset=["execution_date"]).copy()

# ST、*ST、退市股票已在信号截面合并后、中性化和分组之前剔除。
# 不在这里读取 execution_date 的 open / up_limit / down_limit / amount / volume 进行提前过滤。
# 涨跌停、停牌、无成交量等交易约束交给 BigTrader 在实际撮合日处理。

# 每个信号日最终股票等权
selected_df["stock_count"] = selected_df.groupby("signal_date")["instrument"].transform("count")
selected_df = selected_df[selected_df["stock_count"] > 0].copy()
selected_df["target_weight"] = 1.0 / selected_df["stock_count"]

signal_df = selected_df[["signal_date", "execution_date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"]].copy()
signal_df = signal_df.rename(columns={"signal_date": "date"})
signal_df["date"] = pd.to_datetime(signal_df["date"]).dt.strftime("%Y-%m-%d")
signal_df = signal_df.sort_values(["date", "size_group", NEUTRAL_FACTOR_NAME, "instrument"], ascending=[True, True, True, True], kind="mergesort").reset_index(drop=True)

progress_rows("最终交易信号", signal_df)

signal_summary = (
    signal_df.groupby("date")
    .agg(
        execution_date=("execution_date", "first"),
        stock_count=("instrument", "count"),
        avg_weight=("target_weight", "mean")
    )
    .reset_index()
)
progress("交易信号摘要：")
display(signal_summary.head(20))

# BigTrader传入数据。包含所有曾进入目标池的股票，用于引擎订阅和下单。
backtest_data = signal_df[["date", "instrument", "target_weight"]].copy()
backtest_data["date"] = pd.to_datetime(backtest_data["date"]).dt.strftime("%Y-%m-%d")
backtest_data["instrument"] = backtest_data["instrument"].astype(str)

signal_by_date = {
    d: g[["instrument", "target_weight"]].copy()
    for d, g in signal_df.groupby("date")
}

target_by_date = {
    d: set(g["instrument"].astype(str))
    for d, g in signal_df.groupby("date")
}

# 释放不再需要的大表
try:
    del selected_df, status_valid
except Exception:
    pass
gc.collect()


# =========================
# 10. BigTrader 原生回测
# =========================

progress("开始运行 BigTrader 原生回测")


def initialize(context):
    try:
        context.set_commission(
            bigtrader.PerOrder(
                buy_cost=BUY_COST,
                sell_cost=SELL_COST,
                min_cost=MIN_COMMISSION,
            )
        )
    except Exception as e:
        print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

    context.signal_by_date = signal_by_date
    context.target_by_date = target_by_date
    context.rebalance_dates = set(signal_by_date.keys())

    try:
        context.subscribe_bar(list(backtest_data["instrument"].drop_duplicates()), "1d", None)
    except Exception:
        pass


def handle_data(context, data):
    current_date = get_current_date_from_engine(context, data)
    if current_date is None:
        return

    if current_date not in context.rebalance_dates:
        return

    today_signal = context.signal_by_date.get(current_date)
    if today_signal is None or len(today_signal) == 0:
        return

    target_weights = dict(zip(today_signal["instrument"].astype(str), today_signal["target_weight"].astype(float)))
    target_instruments = set(target_weights.keys())
    positions = get_positions_dict(context)
    holding_instruments = set()
    for ins, pos in positions.items():
        if position_amount(pos) > 0:
            holding_instruments.add(str(ins))

    # 先卖出不在目标池中的股票。是否能成交由 BigTrader 在实际撮合日根据停牌、跌停等状态处理。
    for ins in sorted(holding_instruments - target_instruments):
        order_to_target_percent(context, ins, 0.0)

    # 再买入或调整目标股票到等权仓位。是否能成交由 BigTrader 在实际撮合日根据停牌、涨停等状态处理。
    for ins in sorted(target_weights.keys()):
        order_to_target_percent(context, ins, target_weights[ins])


run_kwargs = dict(
    data=backtest_data,
    start_date=START_DATE,
    end_date=END_DATE,
    initialize=initialize,
    handle_data=handle_data,
    capital_base=CAPITAL_BASE,
    benchmark=BENCHMARK,
)

try:
    run_kwargs["market"] = bigtrader.Market.CN_STOCK
except Exception:
    pass

try:
    run_kwargs["frequency"] = bigtrader.Frequency.DAILY
except Exception:
    run_kwargs["frequency"] = "1d"

performance = bigtrader.run(**run_kwargs)

progress("BigTrader 回测完成")

try:
    display(performance.summary)
except Exception:
    display(performance)


从当前的测试结果来看，选择市值组[1,2,3]是较好的选择，同时，在每组当中选择因子值最低的30%的股票可以获得更好的夏普比率和更优的最大回撤，同时带来收益率的提升，但目前的策略回撤还是偏高，这里考虑做与之前同样的防御性+收益补偿策略

## 防御性+收益补偿策略

In [ ]:
# ============================================================
# hml_r_std_5m 市值行业中性化因子策略回测
# 逻辑：全市场按市值分15组，在指定市值组内分别选因子值最低5%的股票，等权持有
# 防御性+收益补偿：中证全指跌破30日均线时，因子股票仓位降至10%，释放仓位等权买入工商银行、交通银行、中国银行；突破30日均线后恢复98%因子股票仓位
# 调仓周期：30个交易日；趋势仓位每日检查，趋势变化时只调整仓位，不重新选股
# 交易约束：信号日剔除指定行业、可识别的ST、*ST、退市股票；停牌和涨跌停由BigTrader成交撮合处理；考虑交易成本
# 严谨性：不在信号日读取下一交易日行情、涨跌停、成交量、停牌等信息，避免潜在未来函数
# 稳定性：固定行业剔除、去重、排序、分组、选股数量规则，降低多次回测的微小差异
# 回测引擎：BigQuant BigTrader 原生回测引擎
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import gc
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e

try:
    from bigquant import bigtrader
except Exception:
    try:
        import bigtrader
    except Exception as e:
        raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 BigTrader。") from e


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

FACTOR_NAME = "hml_r_std_5m"
NEUTRAL_FACTOR_NAME = "factor_mkt_ind_neutral"

# 人工指定参与选股的市值组；1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [15]

N_SIZE_GROUPS = 15
BOTTOM_PCT = 0.2
REBALANCE_DAYS = 30
LOOKBACK_DAYS = 105
MIN_OBS = 80
MIN_STOCKS_PER_SIZE_GROUP = 20

# 行业剔除列表：根据用户图片，保留“非银行金融”，剔除其余列示行业。
# 注意：这里先剔除指定行业，再在剩余股票池内进行市值行业中性化、市值15组分组和选股。
EXCLUDED_INDUSTRIES = [
    "商贸零售",
    "计算机",
    "医药",
    "轻工制造",
    "食品饮料",
    "农林牧渔",
    "传媒",
    "建材",
    "综合",
    "交通运输",
    "通信",
    "银行",
]

# 防止“银行”误伤“非银行金融/非银金融”。
KEEP_INDUSTRIES = ["非银行金融", "非银行业金融", "非银金融"]

# 因子最低5%的取整规则：floor表示向下取整，至少取1只；该规则固定，避免每次理解不一致。
SELECT_COUNT_METHOD = "floor"

WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99

CAPITAL_BASE = 1_000_000
BENCHMARK = "932000.CSI"

# 交易成本：买入佣金 + 卖出佣金与印花税。可按需要调整。
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# =========================
# 防御性 + 收益补偿参数
# =========================

USE_DEFENSIVE_COMPENSATION = True
TREND_INDEX_CODE = "000985.CSI"      # 中证全指
TREND_MA_WINDOW = 30

# 为避免未来函数，t日仓位调整使用 t-1 交易日收盘后已经可知的指数趋势状态。
# 若BigTrader环境确认为“当前K线收盘后下单、下一根K线成交”，也可以改为False；默认保守设为True。
TREND_USE_PREVIOUS_TRADING_DAY = True

RISK_ON_STOCK_EXPOSURE = 0.98        # 指数在30日均线上方：因子目标股票池总仓位98%
RISK_OFF_STOCK_EXPOSURE = 0.10       # 指数跌破30日均线：因子目标股票池总仓位10%
RISK_ON_DEFENSIVE_EXPOSURE = 0.00
RISK_OFF_DEFENSIVE_EXPOSURE = RISK_ON_STOCK_EXPOSURE - RISK_OFF_STOCK_EXPOSURE

DEFENSIVE_BANK_ASSETS = {
    "601398.SH": "工商银行",
    "601328.SH": "交通银行",
    "601988.SH": "中国银行",
}

# 是否优先尝试在SQL侧计算滚动因子，失败后自动切换Python侧计算
USE_SQL_FACTOR_CALC = True

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")
QUERY_START_DATE = (pd.to_datetime(START_DATE) - pd.Timedelta(days=260)).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")

bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if int(g) < 1 or int(g) > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")

SIZE_GROUPS_TO_TRADE = sorted(set([int(x) for x in SIZE_GROUPS_TO_TRADE]))
EXCLUDED_INDUSTRIES = list(dict.fromkeys([str(x).strip() for x in EXCLUDED_INDUSTRIES if str(x).strip()]))
KEEP_INDUSTRIES = list(dict.fromkeys([str(x).strip() for x in KEEP_INDUSTRIES if str(x).strip()]))

if SELECT_COUNT_METHOD not in ["floor", "ceil"]:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


# =========================
# 2. 工具函数
# =========================

_T0 = time.time()


def _elapsed():
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg):
    print(f"[{_elapsed()}] {msg}", flush=True)


def mem_mb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1024 / 1024
    except Exception:
        return np.nan


def progress_rows(name, df):
    m = mem_mb()
    if np.isfinite(m):
        progress(f"{name}：{len(df):,} 行，内存约 {m:.1f} MB")
    else:
        progress(f"{name}：{len(df):,} 行")


def query_df(sql):
    return dai.query(sql).df()


def try_query(sql):
    try:
        df = query_df(sql)
        if df is not None and len(df) > 0:
            return df
    except Exception:
        return None
    return None


def first_success_query(sql_list, err_msg):
    last_error = None
    for sql in sql_list:
        try:
            df = query_df(sql)
            if df is not None and len(df) > 0:
                return df
        except Exception as e:
            last_error = e
            continue
    raise ValueError(f"{err_msg}。最后一次错误：{last_error}")


def date_in_sql(dates):
    return ", ".join([f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates])


def downcast_float(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def winsorize_array(x, q_low=0.01, q_high=0.99):
    x = np.asarray(x, dtype=float)
    lo = np.nanquantile(x, q_low)
    hi = np.nanquantile(x, q_high)
    return np.clip(x, lo, hi)


def zscore_array(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return np.full(len(x), np.nan)
    return (x - mu) / sd


def to_date_str(x):
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def normalize_industry_name(x):
    """将不同来源的行业名称做轻量归一化，便于稳定剔除。"""
    if pd.isna(x):
        return ""
    s = str(x).strip()
    for token in [
        "申万一级行业", "申万一级", "申万", "SW2021", "SW", "中信一级行业", "中信一级",
        "一级行业", "行业", "（", "）", "(", ")", " ", "-", "_"
    ]:
        s = s.replace(token, "")
    alias = {
        "非银金融": "非银行金融",
        "非银行业金融": "非银行金融",
        "商业贸易": "商贸零售",
        "商贸零售业": "商贸零售",
        "食品饮料业": "食品饮料",
        "农林牧渔业": "农林牧渔",
        "轻工制造业": "轻工制造",
        "交通运输业": "交通运输",
        "建筑材料": "建材",
        "银行业": "银行",
        "通信设备": "通信",
        "医药生物": "医药",
    }
    return alias.get(s, s)


EXCLUDED_INDUSTRIES_NORM = set(normalize_industry_name(x) for x in EXCLUDED_INDUSTRIES)
KEEP_INDUSTRIES_NORM = set(normalize_industry_name(x) for x in KEEP_INDUSTRIES)


def is_excluded_industry(x):
    """判断行业是否在剔除名单内；非银行金融显式保留，避免被“银行”误伤。"""
    s = normalize_industry_name(x)
    if not s:
        return False
    if s in KEEP_INDUSTRIES_NORM:
        return False
    if "非银行金融" in s or "非银金融" in s or "非银行业金融" in s:
        return False
    if s in EXCLUDED_INDUSTRIES_NORM:
        return True
    # 兼容“xx银行”等名称，但仍不匹配非银行金融。
    if "银行" in s and "非银行" not in s and "非银" not in s:
        return True
    for ex in EXCLUDED_INDUSTRIES_NORM:
        if ex == "银行":
            continue
        if ex and (ex in s or s in ex):
            return True
    return False


def calc_select_count(n, pct):
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def get_current_date_from_engine(context, data):
    if data is not None and hasattr(data, "current_dt"):
        return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            v = getattr(context, attr)
            if v is not None:
                return pd.to_datetime(v).strftime("%Y-%m-%d")
    return None


def get_positions_dict(context):
    for method in ["get_account_positions", "get_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    return {}


def position_amount(pos_obj):
    try:
        return float(getattr(pos_obj, "amount", 0))
    except Exception:
        try:
            return float(pos_obj.get("amount", 0))
        except Exception:
            return 0.0


def order_to_target_percent(context, instrument, weight):
    try:
        context.order_target_percent(instrument, float(weight))
        return True
    except Exception:
        try:
            # 兼容部分文档中的 order_percent 写法；在BigTrader股票策略中通常也表示目标仓位比例。
            context.order_percent(instrument, float(weight))
            return True
        except Exception as e:
            print(f"下单失败：{instrument}, target={weight:.6f}, err={e}", flush=True)
            return False


def to_bigtrader_instrument(inst):
    """兼容不同BigQuant环境中的证券代码后缀。"""
    s = str(inst)
    if s.endswith(".SZA"):
        return s[:-4] + ".SZ"
    if s.endswith(".SHA"):
        return s[:-4] + ".SH"
    if s.endswith(".BJA"):
        return s[:-4] + ".BJ"
    return s


def build_trend_allocation_df(trade_dates, query_start_date, end_date):
    """
    构造每日趋势仓位序列。

    严谨性：
    - 趋势信号只使用中证全指已经发生的收盘价和30日均线；
    - 默认 t 日调仓使用 t-1 交易日的趋势状态，避免在开盘调仓时读取 t 日收盘价；
    - 若指数数据缺失，直接报错，不用未来数据填补。
    """
    trade_dates = pd.DatetimeIndex(pd.to_datetime(trade_dates)).sort_values()
    if len(trade_dates) == 0:
        raise ValueError("trade_dates 为空，无法构造趋势仓位序列。")

    if not USE_DEFENSIVE_COMPENSATION:
        return pd.DataFrame(
            {
                "stock_exposure": float(RISK_ON_STOCK_EXPOSURE),
                "defensive_exposure": float(RISK_ON_DEFENSIVE_EXPOSURE),
                "risk_on": True,
            },
            index=trade_dates,
        )

    index_sql_candidates = [
        f"""
        SELECT date, instrument, close
        FROM cn_stock_index_bar1d
        WHERE instrument = '{TREND_INDEX_CODE}'
          AND date >= '{query_start_date}'
          AND date <= '{end_date}'
        ORDER BY date
        """,
        f"""
        SELECT date, instrument, close
        FROM cn_stock_bar1d
        WHERE instrument = '{TREND_INDEX_CODE}'
          AND date >= '{query_start_date}'
          AND date <= '{end_date}'
        ORDER BY date
        """,
    ]

    idx = first_success_query(index_sql_candidates, f"无法读取趋势过滤指数 {TREND_INDEX_CODE} 的日线收盘价")
    idx["date"] = pd.to_datetime(idx["date"])
    idx["close"] = pd.to_numeric(idx["close"], errors="coerce")
    idx = idx.dropna(subset=["date", "close"])
    idx = idx[idx["close"] > 0].copy()
    idx = idx.sort_values("date", kind="mergesort").drop_duplicates("date", keep="last")

    if len(idx) < TREND_MA_WINDOW:
        raise ValueError(f"指数数据不足，无法计算 {TREND_MA_WINDOW} 日均线。")

    close = idx.set_index("date")["close"].sort_index()
    ma = close.rolling(window=TREND_MA_WINDOW, min_periods=TREND_MA_WINDOW).mean()
    risk_on_raw = (close >= ma)
    risk_on_raw = risk_on_raw.where(ma.notna(), True)

    raw_alloc = pd.DataFrame(
        {
            "risk_on_raw": risk_on_raw.astype(bool),
            "stock_exposure_raw": np.where(risk_on_raw, RISK_ON_STOCK_EXPOSURE, RISK_OFF_STOCK_EXPOSURE),
            "defensive_exposure_raw": np.where(risk_on_raw, RISK_ON_DEFENSIVE_EXPOSURE, RISK_OFF_DEFENSIVE_EXPOSURE),
        },
        index=close.index,
    )

    raw_alloc = raw_alloc.reindex(trade_dates).ffill()

    if TREND_USE_PREVIOUS_TRADING_DAY:
        risk_on = raw_alloc["risk_on_raw"].shift(1).fillna(True).astype(bool)
        stock_exposure = raw_alloc["stock_exposure_raw"].shift(1).fillna(float(RISK_ON_STOCK_EXPOSURE))
        defensive_exposure = raw_alloc["defensive_exposure_raw"].shift(1).fillna(float(RISK_ON_DEFENSIVE_EXPOSURE))
    else:
        risk_on = raw_alloc["risk_on_raw"].fillna(True).astype(bool)
        stock_exposure = raw_alloc["stock_exposure_raw"].fillna(float(RISK_ON_STOCK_EXPOSURE))
        defensive_exposure = raw_alloc["defensive_exposure_raw"].fillna(float(RISK_ON_DEFENSIVE_EXPOSURE))

    out = pd.DataFrame(
        {
            "stock_exposure": stock_exposure.astype(float),
            "defensive_exposure": defensive_exposure.astype(float),
            "risk_on": risk_on.astype(bool),
        },
        index=trade_dates,
    )
    return out


# =========================
# 3. 获取交易日和调仓信号日期
# =========================

progress("开始获取交易日列表")
trade_dates_sql = f"""
SELECT DISTINCT date
FROM cn_stock_bar1d
WHERE date >= '{START_DATE}'
  AND date <= '{END_DATE}'
ORDER BY date
"""
trade_dates_df = query_df(trade_dates_sql)
trade_dates_df["date"] = pd.to_datetime(trade_dates_df["date"])
trade_dates = trade_dates_df["date"].drop_duplicates().sort_values().reset_index(drop=True)

if len(trade_dates) < REBALANCE_DAYS + 2:
    raise ValueError("指定时间段内交易日过少，无法完成30日调仓回测。")

signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()

# BigTrader日线回测通常在当前K线提交订单，并在下一根K线撮合成交；
# 因此这里用 signal_date 计算因子和组合，订单实际在 next_trade_date 交易。
signal_to_execution = {}
for dt in signal_dates:
    idx = trade_dates[trade_dates == dt].index
    if len(idx) == 0:
        continue
    next_idx = int(idx[0]) + 1
    if next_idx < len(trade_dates):
        signal_to_execution[to_date_str(dt)] = to_date_str(trade_dates.iloc[next_idx])

signal_dates = [pd.to_datetime(k) for k in signal_to_execution.keys()]
if len(signal_dates) == 0:
    raise ValueError("没有可用调仓信号日期。")

signal_date_sql = date_in_sql(signal_dates)
progress(f"交易日数量：{len(trade_dates):,}；调仓信号截面数量：{len(signal_dates):,}")
progress(f"回测使用市值组：{SIZE_GROUPS_TO_TRADE}；每组选取最低 {BOTTOM_PCT:.1%}；选股数量取整：{SELECT_COUNT_METHOD}")
progress(f"剔除行业：{EXCLUDED_INDUSTRIES}；显式保留行业：{KEEP_INDUSTRIES}")

progress("开始构造中证全指30日均线防御仓位序列")
trend_allocation_df = build_trend_allocation_df(trade_dates, QUERY_START_DATE, END_DATE)
trend_summary = trend_allocation_df.loc[
    (trend_allocation_df.index >= pd.to_datetime(START_DATE)) &
    (trend_allocation_df.index <= pd.to_datetime(END_DATE))
].copy()
risk_on_days = int(trend_summary["risk_on"].sum())
risk_off_days = int((~trend_summary["risk_on"]).sum())
progress(
    f"趋势仓位序列完成：风险开启 {risk_on_days:,} 天，风险关闭 {risk_off_days:,} 天；"
    f"风险开启股票仓位 {RISK_ON_STOCK_EXPOSURE:.2%}，风险关闭股票仓位 {RISK_OFF_STOCK_EXPOSURE:.2%}，"
    f"风险关闭银行仓位 {RISK_OFF_DEFENSIVE_EXPOSURE:.2%}"
)

del trade_dates_df
# trade_dates 后续仍用于构造BigTrader每日触发数据，不删除。
gc.collect()


# =========================
# 4. 计算 hml_r_std_5m 因子
# =========================

progress("开始计算 hml_r_std_5m 因子")

factor_df = None
if USE_SQL_FACTOR_CALC:
    sql_factor_candidates = [
        f"""
        WITH base AS (
            SELECT
                date,
                instrument,
                high,
                low,
                close,
                pre_close
            FROM cn_stock_bar1d
            WHERE date >= '{QUERY_START_DATE}'
              AND date <= '{END_DATE}'
              AND high > 0
              AND low > 0
              AND close > 0
        ), ret AS (
            SELECT
                date,
                instrument,
                high / pre_close - 1 AS high_r,
                low / pre_close - 1 AS low_r
            FROM base
            WHERE pre_close > 0
        ), fac AS (
            SELECT
                date,
                instrument,
                stddev_samp(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS high_r_std_5m,
                stddev_samp(low_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS low_r_std_5m,
                count(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS obs_cnt
            FROM ret
        )
        SELECT
            date,
            instrument,
            high_r_std_5m - low_r_std_5m AS {FACTOR_NAME}
        FROM fac
        WHERE date IN ({signal_date_sql})
          AND obs_cnt >= {MIN_OBS}
        """,
        f"""
        WITH base AS (
            SELECT
                date,
                instrument,
                high,
                low,
                close,
                lag(close) OVER (PARTITION BY instrument ORDER BY date) AS pre_close
            FROM cn_stock_bar1d
            WHERE date >= '{QUERY_START_DATE}'
              AND date <= '{END_DATE}'
              AND high > 0
              AND low > 0
              AND close > 0
        ), ret AS (
            SELECT
                date,
                instrument,
                high / pre_close - 1 AS high_r,
                low / pre_close - 1 AS low_r
            FROM base
            WHERE pre_close > 0
        ), fac AS (
            SELECT
                date,
                instrument,
                stddev_samp(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS high_r_std_5m,
                stddev_samp(low_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS low_r_std_5m,
                count(high_r) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN {LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
                ) AS obs_cnt
            FROM ret
        )
        SELECT
            date,
            instrument,
            high_r_std_5m - low_r_std_5m AS {FACTOR_NAME}
        FROM fac
        WHERE date IN ({signal_date_sql})
          AND obs_cnt >= {MIN_OBS}
        """,
    ]
    try:
        factor_df = first_success_query(
            sql_factor_candidates,
            "SQL侧计算 hml_r_std_5m 失败，可能是表字段或窗口函数不兼容"
        )
    except Exception as e:
        progress(f"SQL侧计算失败，切换为 Python 侧滚动计算。原因：{e}")
        factor_df = None

if factor_df is None:
    progress("开始 Python 侧滚动计算")
    bar_sql_candidates = [
        f"""
        SELECT date, instrument, high, low, close, pre_close
        FROM cn_stock_bar1d
        WHERE date >= '{QUERY_START_DATE}'
          AND date <= '{END_DATE}'
        ORDER BY instrument, date
        """,
        f"""
        SELECT date, instrument, high, low, close
        FROM cn_stock_bar1d
        WHERE date >= '{QUERY_START_DATE}'
          AND date <= '{END_DATE}'
        ORDER BY instrument, date
        """,
    ]
    bar = first_success_query(bar_sql_candidates, "无法从 cn_stock_bar1d 读取计算因子所需字段")
    bar["date"] = pd.to_datetime(bar["date"])
    bar = downcast_float(bar, ["high", "low", "close", "pre_close"])
    bar = bar.dropna(subset=["date", "instrument", "high", "low", "close"])
    bar = bar[(bar["high"] > 0) & (bar["low"] > 0) & (bar["close"] > 0)].copy()
    bar = bar.sort_values(["instrument", "date"]).reset_index(drop=True)
    progress_rows("Python侧日行情数据", bar)

    if "pre_close" not in bar.columns or bar["pre_close"].isna().all():
        bar["pre_close"] = bar.groupby("instrument", sort=False)["close"].shift(1)

    bar["high_r"] = bar["high"] / bar["pre_close"] - 1
    bar["low_r"] = bar["low"] / bar["pre_close"] - 1
    bar.loc[~np.isfinite(bar["high_r"]), "high_r"] = np.nan
    bar.loc[~np.isfinite(bar["low_r"]), "low_r"] = np.nan

    bar["high_r_std_5m"] = (
        bar.groupby("instrument", sort=False)["high_r"]
        .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
        .std()
        .reset_index(level=0, drop=True)
    )
    bar["low_r_std_5m"] = (
        bar.groupby("instrument", sort=False)["low_r"]
        .rolling(window=LOOKBACK_DAYS, min_periods=MIN_OBS)
        .std()
        .reset_index(level=0, drop=True)
    )
    bar[FACTOR_NAME] = bar["high_r_std_5m"] - bar["low_r_std_5m"]
    factor_df = bar.loc[bar["date"].isin(signal_dates), ["date", "instrument", FACTOR_NAME]].copy()
    del bar
    gc.collect()

factor_df["date"] = pd.to_datetime(factor_df["date"])
factor_df = downcast_float(factor_df, [FACTOR_NAME])
factor_df = factor_df.dropna(subset=["date", "instrument", FACTOR_NAME])
factor_df = factor_df[np.isfinite(factor_df[FACTOR_NAME])].copy()
factor_df = factor_df.sort_values(["date", "instrument"], kind="mergesort")
factor_df = factor_df.drop_duplicates(subset=["date", "instrument"], keep="last")
progress_rows("调仓截面因子数据", factor_df)


# =========================
# 5. 读取调仓截面市值数据
# =========================

progress("开始读取调仓截面市值数据")

mkt_table_candidates = [
    "cn_stock_valuation",
    "cn_stock_prefactors",
    "cn_stock_factors",
    "cn_stock_derivative_indicator",
    "cn_stock_bar1d",
]
mkt_date_candidates = ["date", "trade_date"]
mkt_instrument_candidates = ["instrument", "order_book_id", "ts_code", "symbol"]
mkt_col_candidates = [
    "total_market_cap",
    "market_cap",
    "total_mv",
    "mkt_cap",
    "total_market_value",
    "circ_market_cap",
    "float_market_cap",
    "circ_mv",
    "float_mv",
]

mkt = None
mkt_source = None
for table in mkt_table_candidates:
    for date_col in mkt_date_candidates:
        for instrument_col in mkt_instrument_candidates:
            for mkt_col in mkt_col_candidates:
                sql = f"""
                SELECT
                    {date_col} AS date,
                    {instrument_col} AS instrument,
                    {mkt_col} AS mkt_cap
                FROM {table}
                WHERE {date_col} IN ({signal_date_sql})
                """
                tmp = try_query(sql)
                if tmp is not None:
                    mkt = tmp.copy()
                    mkt_source = f"{table}.{mkt_col}"
                    break
            if mkt is not None:
                break
        if mkt is not None:
            break
    if mkt is not None:
        break

if mkt is None or len(mkt) == 0:
    raise ValueError("未能读取市值字段。请确认平台中可用的市值表或字段名称。")

mkt["date"] = pd.to_datetime(mkt["date"])
mkt["mkt_cap"] = pd.to_numeric(mkt["mkt_cap"], errors="coerce")
mkt = mkt.dropna(subset=["date", "instrument", "mkt_cap"])
mkt = mkt[mkt["mkt_cap"] > 0].copy()
mkt = mkt.sort_values(["date", "instrument"], kind="mergesort")
mkt = mkt.drop_duplicates(subset=["date", "instrument"], keep="last")
mkt = downcast_float(mkt, ["mkt_cap"])
progress_rows(f"市值数据来源 {mkt_source}", mkt)


# =========================
# 6. 读取调仓截面行业数据
# =========================

progress("开始读取行业数据")

industry_table_candidates = [
    "cn_stock_industry_component",
    "cn_stock_industry",
    "cn_stock_industry_classification",
    "cn_stock_basic_info",
    "cn_stock_prefactors",
    "cn_stock_factors",
]
industry_date_candidates = ["date", "trade_date"]
industry_instrument_candidates = ["instrument", "order_book_id", "ts_code", "symbol"]
industry_col_candidates = [
    "sw2021_level1",
    "sw_level1",
    "sw_l1",
    "industry_level1",
    "industry_name_level1",
    "industry_l1",
    "industry_name",
    "industry",
    "sector",
]

industry = None
industry_source = None
industry_has_date = False

for table in industry_table_candidates:
    for date_col in industry_date_candidates:
        for instrument_col in industry_instrument_candidates:
            for industry_col in industry_col_candidates:
                sql = f"""
                SELECT
                    {date_col} AS date,
                    {instrument_col} AS instrument,
                    {industry_col} AS industry
                FROM {table}
                WHERE {date_col} IN ({signal_date_sql})
                """
                tmp = try_query(sql)
                if tmp is not None:
                    industry = tmp.copy()
                    industry_source = f"{table}.{industry_col}"
                    industry_has_date = True
                    break
            if industry is not None:
                break
        if industry is not None:
            break
    if industry is not None:
        break

if industry is None:
    for table in industry_table_candidates:
        for instrument_col in industry_instrument_candidates:
            for industry_col in industry_col_candidates:
                sql = f"""
                SELECT
                    {instrument_col} AS instrument,
                    {industry_col} AS industry
                FROM {table}
                """
                tmp = try_query(sql)
                if tmp is not None:
                    industry = tmp.copy()
                    industry_source = f"{table}.{industry_col}"
                    industry_has_date = False
                    break
            if industry is not None:
                break
        if industry is not None:
            break

if industry is None or len(industry) == 0:
    raise ValueError("未能读取行业字段。请确认平台中可用的行业表或字段名称。")

industry["industry"] = industry["industry"].astype(str)
industry = industry[industry["industry"].notna() & (industry["industry"] != "nan")].copy()

if industry_has_date:
    industry["date"] = pd.to_datetime(industry["date"])
    industry = industry.sort_values(["date", "instrument", "industry"], kind="mergesort")
    industry = industry.drop_duplicates(subset=["date", "instrument"], keep="last")
else:
    industry = industry.sort_values(["instrument", "industry"], kind="mergesort")
    industry = industry.drop_duplicates(subset=["instrument"], keep="last")

progress_rows(f"行业数据来源 {industry_source}", industry)


# =========================
# 7. 读取信号日 ST、退市状态数据
# =========================
# 严谨性说明：
# - 这里只尝试读取带日期的状态字段，并且只使用 signal_date 当日信息；
# - 不使用不带日期的静态证券名称或当前ST状态做历史过滤，避免把未来状态带入历史回测；
# - 若平台无法提供带日期状态字段，则跳过显式ST过滤，由BigTrader交易数据和撮合规则处理不可交易证券。

progress("开始读取信号日 ST、退市状态数据")

status_table_candidates = [
    "cn_stock_bar1d",
    "cn_stock_status",
    "cn_stock_basic_info",
    "cn_stock_instruments",
    "cn_stock_description",
]
status_date_candidates = ["date", "trade_date"]
status_instrument_candidates = ["instrument", "order_book_id", "ts_code", "symbol"]
name_col_candidates = ["name", "short_name", "sec_name", "security_name", "display_name"]
st_col_candidates = ["is_st", "st", "is_special_treatment", "special_treatment"]
list_status_candidates = ["list_status", "status", "listed_state", "delist_status"]

status = None
status_source = None
status_has_date = False

# 优先：带日期 + 名称 + 可选ST字段
for table in status_table_candidates:
    for date_col in status_date_candidates:
        for instrument_col in status_instrument_candidates:
            for name_col in name_col_candidates:
                for st_col in [None] + st_col_candidates:
                    select_cols = [
                        f"{date_col} AS date",
                        f"{instrument_col} AS instrument",
                        f"{name_col} AS name",
                    ]
                    if st_col is not None:
                        select_cols.append(f"{st_col} AS is_st_flag")
                    sql = f"""
                    SELECT {', '.join(select_cols)}
                    FROM {table}
                    WHERE {date_col} IN ({signal_date_sql})
                    """
                    tmp = try_query(sql)
                    if tmp is not None:
                        status = tmp.copy()
                        status_source = f"{table}.{name_col}" + (f"+{st_col}" if st_col is not None else "")
                        status_has_date = True
                        break
                if status is not None:
                    break
            if status is not None:
                break
        if status is not None:
            break
    if status is not None:
        break

# 备选：带日期 + ST字段，不要求名称字段
if status is None:
    for table in status_table_candidates:
        for date_col in status_date_candidates:
            for instrument_col in status_instrument_candidates:
                for st_col in st_col_candidates:
                    sql = f"""
                    SELECT
                        {date_col} AS date,
                        {instrument_col} AS instrument,
                        {st_col} AS is_st_flag
                    FROM {table}
                    WHERE {date_col} IN ({signal_date_sql})
                    """
                    tmp = try_query(sql)
                    if tmp is not None:
                        status = tmp.copy()
                        status_source = f"{table}.{st_col}"
                        status_has_date = True
                        break
                if status is not None:
                    break
            if status is not None:
                break
        if status is not None:
            break

# 备选：带日期 + 上市状态字段
if status is None:
    for table in status_table_candidates:
        for date_col in status_date_candidates:
            for instrument_col in status_instrument_candidates:
                for list_status_col in list_status_candidates:
                    sql = f"""
                    SELECT
                        {date_col} AS date,
                        {instrument_col} AS instrument,
                        {list_status_col} AS list_status
                    FROM {table}
                    WHERE {date_col} IN ({signal_date_sql})
                    """
                    tmp = try_query(sql)
                    if tmp is not None:
                        status = tmp.copy()
                        status_source = f"{table}.{list_status_col}"
                        status_has_date = True
                        break
                if status is not None:
                    break
            if status is not None:
                break
        if status is not None:
            break

if status is not None and len(status) > 0 and status_has_date:
    status["date"] = pd.to_datetime(status["date"])

    bad = pd.Series(False, index=status.index)

    if "name" in status.columns:
        name_s = status["name"].astype(str)
        bad = bad | name_s.str.contains("ST", case=False, na=False)
        bad = bad | name_s.str.contains("退", na=False)

    if "is_st_flag" in status.columns:
        st_flag = pd.to_numeric(status["is_st_flag"], errors="coerce").fillna(0) != 0
        bad = bad | st_flag

    if "list_status" in status.columns:
        ls = status["list_status"].astype(str).str.lower()
        bad = bad | ls.isin([
            "delisted", "de_listed", "退市", "已退市", "terminated",
            "d", "0", "false", "暂停上市", "终止上市"
        ])

    status["is_bad_status"] = bad.astype(bool)
    status_valid = status[["date", "instrument", "is_bad_status"]].copy()
    status_valid = status_valid.sort_values(["date", "instrument", "is_bad_status"], kind="mergesort")
    status_valid = status_valid.drop_duplicates(subset=["date", "instrument"], keep="last")
    progress_rows(f"状态数据来源 {status_source}", status_valid)
else:
    progress("警告：未读取到可按信号日对齐的ST/退市状态字段；为避免未来函数，不使用静态当前名称或当前状态过滤。")
    status_valid = None
    status_has_date = False


# =========================
# 8. 合并信号截面数据
# =========================

progress("开始合并信号截面数据")

signal_panel = factor_df.merge(mkt, on=["date", "instrument"], how="inner")
del factor_df, mkt
gc.collect()

if industry_has_date:
    signal_panel = signal_panel.merge(industry[["date", "instrument", "industry"]], on=["date", "instrument"], how="inner")
else:
    signal_panel = signal_panel.merge(industry[["instrument", "industry"]], on="instrument", how="inner")
del industry
gc.collect()

signal_panel = signal_panel.dropna(subset=[FACTOR_NAME, "mkt_cap", "industry"])
signal_panel = signal_panel[(signal_panel["mkt_cap"] > 0) & np.isfinite(signal_panel[FACTOR_NAME])].copy()
signal_panel = signal_panel.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)

# 只使用信号日可得的状态字段剔除 ST、*ST、退市股票；该步骤放在中性化和市值分组之前，统一股票池口径。
if status_valid is not None:
    before_n = len(signal_panel)
    signal_panel = signal_panel.merge(
        status_valid[["date", "instrument", "is_bad_status"]],
        on=["date", "instrument"],
        how="left"
    )
    signal_panel["is_bad_status"] = signal_panel["is_bad_status"].fillna(False).astype(bool)
    bad_n = int(signal_panel["is_bad_status"].sum())
    signal_panel = signal_panel[~signal_panel["is_bad_status"]].drop(columns=["is_bad_status"]).copy()
    progress(f"信号日ST/退市剔除：{bad_n:,} 行；剩余 {len(signal_panel):,} 行；剔除前 {before_n:,} 行")
else:
    progress("未使用ST/退市状态过滤：平台未提供可按信号日对齐的状态字段。")

# 剔除指定行业，显式保留非银行金融。
signal_panel["industry_raw"] = signal_panel["industry"].astype(str)
signal_panel["industry_norm"] = signal_panel["industry_raw"].map(normalize_industry_name)
industry_excluded_mask = signal_panel["industry_raw"].map(is_excluded_industry).astype(bool)
excluded_n = int(industry_excluded_mask.sum())
progress(f"行业剔除：{excluded_n:,} 行；剩余 {len(signal_panel) - excluded_n:,} 行")
if excluded_n > 0:
    excluded_dist = (
        signal_panel.loc[industry_excluded_mask, "industry_norm"]
        .value_counts()
        .head(20)
        .reset_index()
    )
    excluded_dist.columns = ["剔除行业", "样本行数"]
    progress("剔除行业样本分布前20：")
    display(excluded_dist)

signal_panel = signal_panel[~industry_excluded_mask].copy()
if len(signal_panel) == 0:
    raise ValueError("行业与状态过滤后没有有效信号样本，请检查剔除行业列表或行业字段。")

signal_panel["log_mkt_cap"] = np.log(signal_panel["mkt_cap"].astype(float))
signal_panel["industry"] = signal_panel["industry_norm"].astype("category")
progress_rows("合并、状态过滤、行业剔除后的信号截面数据", signal_panel)

if len(signal_panel) == 0:
    raise ValueError("合并后没有有效信号样本，请检查行情、市值或行业字段匹配。")


# =========================
# 9. 市值行业中性化、15档市值分组、选股
# =========================
# 严谨性说明：
# - 因子只使用 signal_date 及之前的日频数据；
# - 市值和行业只使用 signal_date 的截面信息；
# - 订单在 signal_date 生成，BigTrader日线机制下通常于下一根K线成交；
# - 不在 signal_date 读取下一交易日行情、成交量、停牌、涨跌停等信息；
# - 涨跌停、停牌导致的无法成交由 BigTrader 撮合机制处理；
# - ST、退市过滤只使用 signal_date 可得的带日期状态字段。

progress("开始逐截面市值行业中性化、15档市值分组与选股")

unique_industry_count = signal_panel["industry"].astype(str).nunique(dropna=True)
progress(f"行业字段去重数量：{unique_industry_count:,}")
if unique_industry_count > 120:
    progress("警告：行业字段去重数量偏高，可能读取到的不是一级行业字段；代码仍会继续。")


def neutralize_group_and_select(g):
    g = g[["date", "instrument", FACTOR_NAME, "mkt_cap", "log_mkt_cap", "industry"]].copy()
    g = g.sort_values(["instrument"], kind="mergesort").reset_index(drop=True)
    if len(g) < max(100, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    g[FACTOR_NAME] = pd.to_numeric(g[FACTOR_NAME], errors="coerce")
    g["mkt_cap"] = pd.to_numeric(g["mkt_cap"], errors="coerce")
    g["log_mkt_cap"] = pd.to_numeric(g["log_mkt_cap"], errors="coerce")
    g["industry"] = g["industry"].astype(str)
    g = g.dropna(subset=[FACTOR_NAME, "mkt_cap", "log_mkt_cap", "industry"])
    g = g[(g["mkt_cap"] > 0) & np.isfinite(g[FACTOR_NAME]) & np.isfinite(g["log_mkt_cap"])].copy()

    if len(g) < max(100, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    # 截面去极值
    g["factor_w"] = winsorize_array(g[FACTOR_NAME].values, WINSOR_Q_LOW, WINSOR_Q_HIGH)

    # 市值标准化后，用FWL方式剔除行业固定效应和市值暴露
    g["log_mkt_z"] = zscore_array(g["log_mkt_cap"].values)
    g = g.dropna(subset=["factor_w", "log_mkt_z"])
    if len(g) < max(100, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    ind_group = g.groupby("industry", sort=True, observed=True)
    y_dm = g["factor_w"] - ind_group["factor_w"].transform("mean")
    x_dm = g["log_mkt_z"] - ind_group["log_mkt_z"].transform("mean")

    y = y_dm.astype(float).values
    x = x_dm.astype(float).values
    valid = np.isfinite(y) & np.isfinite(x)

    if valid.sum() < 100:
        return None

    y_valid = y[valid]
    x_valid = x[valid]
    x_var = np.sum(x_valid ** 2)
    if x_var <= 0 or not np.isfinite(x_var):
        return None

    beta = np.sum(x_valid * y_valid) / x_var
    resid = y_valid - beta * x_valid

    out = g.loc[valid, ["date", "instrument", "mkt_cap"]].copy()
    out[NEUTRAL_FACTOR_NAME] = zscore_array(resid)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "mkt_cap"])

    # 固定排序后再rank，保证市值相同或近似相同时的分组结果稳定。
    out = out.sort_values(["mkt_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = out["mkt_cap"].rank(method="first", ascending=True)
    try:
        out["size_group"] = pd.qcut(
            rank,
            q=N_SIZE_GROUPS,
            labels=list(range(1, N_SIZE_GROUPS + 1))
        ).astype(int)
    except Exception:
        return None

    selected_parts = []
    for size_group in SIZE_GROUPS_TO_TRADE:
        sg = out[out["size_group"] == size_group].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue
        n_select = calc_select_count(len(sg), BOTTOM_PCT)
        sg = sg.sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[True, True], kind="mergesort")
        selected_parts.append(sg.head(n_select))

    if not selected_parts:
        return None

    selected = pd.concat(selected_parts, ignore_index=True)
    return selected[["date", "instrument", "size_group", NEUTRAL_FACTOR_NAME]]


selected_parts = []
all_signal_dates = sorted(signal_panel["date"].drop_duplicates())
for i, dt in enumerate(all_signal_dates, 1):
    g = signal_panel.loc[signal_panel["date"] == dt]
    if i == 1 or i % 5 == 0 or i == len(all_signal_dates):
        progress(f"选股进度：{i}/{len(all_signal_dates)}，信号日 {to_date_str(dt)}，样本 {len(g):,}")
    sel = neutralize_group_and_select(g)
    if sel is not None and len(sel) > 0:
        selected_parts.append(sel)

if not selected_parts:
    raise ValueError("没有形成任何有效选股结果，请检查参数或数据。")

selected_df = pd.concat(selected_parts, ignore_index=True)
del selected_parts, signal_panel
gc.collect()
progress_rows("初步选股结果", selected_df)

selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
selected_df = selected_df.dropna(subset=["execution_date"]).copy()

# ST、*ST、退市股票已在信号截面合并后、中性化和分组之前剔除。
# 不在这里读取 execution_date 的 open / up_limit / down_limit / amount / volume 进行提前过滤。
# 涨跌停、停牌、无成交量等交易约束交给 BigTrader 在实际撮合日处理。

# 每个信号日最终股票等权
selected_df["stock_count"] = selected_df.groupby("signal_date")["instrument"].transform("count")
selected_df = selected_df[selected_df["stock_count"] > 0].copy()
selected_df["target_weight"] = 1.0 / selected_df["stock_count"]

signal_df = selected_df[["signal_date", "execution_date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"]].copy()
signal_df = signal_df.rename(columns={"signal_date": "date"})
signal_df["date"] = pd.to_datetime(signal_df["date"]).dt.strftime("%Y-%m-%d")
signal_df = signal_df.sort_values(["date", "size_group", NEUTRAL_FACTOR_NAME, "instrument"], ascending=[True, True, True, True], kind="mergesort").reset_index(drop=True)

progress_rows("最终交易信号", signal_df)

signal_summary = (
    signal_df.groupby("date")
    .agg(
        execution_date=("execution_date", "first"),
        stock_count=("instrument", "count"),
        avg_weight=("target_weight", "mean")
    )
    .reset_index()
)
progress("交易信号摘要：")
display(signal_summary.head(20))

# BigTrader传入数据。为确保趋势变化可以每日响应，这里构造“每日×订阅标的”的轻量触发数据。
DEFENSIVE_BANK_INSTRUMENTS = [to_bigtrader_instrument(x) for x in DEFENSIVE_BANK_ASSETS.keys()]

signal_df["instrument"] = signal_df["instrument"].map(to_bigtrader_instrument)

signal_by_date = {
    d: g[["instrument", "target_weight"]].drop_duplicates("instrument").copy()
    for d, g in signal_df.groupby("date")
}

target_by_date = {
    d: set(g["instrument"].astype(str))
    for d, g in signal_df.groupby("date")
}

all_backtest_instruments = sorted(
    set(signal_df["instrument"].dropna().astype(str).unique().tolist()) |
    set(DEFENSIVE_BANK_INSTRUMENTS)
)

backtest_dates = pd.to_datetime(trade_dates)
backtest_dates = backtest_dates[
    (backtest_dates >= pd.to_datetime(START_DATE)) &
    (backtest_dates <= pd.to_datetime(END_DATE))
]
backtest_date_strings = [to_date_str(x) for x in backtest_dates]

# 行数约为“交易日数 × 策略涉及股票数”，只包含 date/instrument/target_weight 三列，换取每日趋势响应能力。
backtest_data = pd.MultiIndex.from_product(
    [backtest_date_strings, all_backtest_instruments],
    names=["date", "instrument"]
).to_frame(index=False)
backtest_data["target_weight"] = np.float32(0.0)
progress_rows("BigTrader每日触发订阅数据", backtest_data)

trend_allocation_for_bt = trend_allocation_df.loc[
    (trend_allocation_df.index >= pd.to_datetime(START_DATE)) &
    (trend_allocation_df.index <= pd.to_datetime(END_DATE))
].copy()

stock_exposure_by_date = {
    to_date_str(d): float(row["stock_exposure"])
    for d, row in trend_allocation_for_bt.iterrows()
}

defensive_exposure_by_date = {
    to_date_str(d): float(row["defensive_exposure"])
    for d, row in trend_allocation_for_bt.iterrows()
}

risk_on_by_date = {
    to_date_str(d): bool(row["risk_on"])
    for d, row in trend_allocation_for_bt.iterrows()
}

progress("防御资产：" + "、".join([f"{name}({to_bigtrader_instrument(code)})" for code, name in DEFENSIVE_BANK_ASSETS.items()]))
progress(f"BigTrader订阅标的数量：{len(all_backtest_instruments):,}；每日趋势仓位日期数量：{len(stock_exposure_by_date):,}")

try:
    del selected_df, status_valid, trend_allocation_df, trend_allocation_for_bt
except Exception:
    pass
gc.collect()


# =========================
# 10. BigTrader 原生回测
# =========================

progress("开始运行 BigTrader 原生回测")


def initialize(context):
    try:
        context.set_commission(
            bigtrader.PerOrder(
                buy_cost=BUY_COST,
                sell_cost=SELL_COST,
                min_cost=MIN_COMMISSION,
            )
        )
    except Exception as e:
        print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

    context.signal_by_date = signal_by_date
    context.target_by_date = target_by_date
    context.rebalance_dates = set(signal_by_date.keys())

    context.stock_exposure_by_date = stock_exposure_by_date
    context.defensive_exposure_by_date = defensive_exposure_by_date
    context.risk_on_by_date = risk_on_by_date

    context.defensive_bank_instruments = DEFENSIVE_BANK_INSTRUMENTS
    context.defensive_bank_names = {
        to_bigtrader_instrument(code): name
        for code, name in DEFENSIVE_BANK_ASSETS.items()
    }

    context.current_factor_targets = []
    context.current_target_date = None
    context.current_stock_exposure = None
    context.current_defensive_exposure = None

    try:
        context.subscribe_bar(all_backtest_instruments, "1d", None)
    except Exception:
        pass

    print("initialize 完成：因子选股 + 中证全指30日均线防御性收益补偿策略", flush=True)
    print("防御资产：" + "、".join([f"{context.defensive_bank_names.get(x, x)}({x})" for x in context.defensive_bank_instruments]), flush=True)


def _get_stock_exposure(context, date_str):
    try:
        return float(context.stock_exposure_by_date.get(date_str, RISK_ON_STOCK_EXPOSURE))
    except Exception:
        return float(RISK_ON_STOCK_EXPOSURE)


def _get_defensive_exposure(context, date_str):
    try:
        return float(context.defensive_exposure_by_date.get(date_str, RISK_ON_DEFENSIVE_EXPOSURE))
    except Exception:
        return float(RISK_ON_DEFENSIVE_EXPOSURE)


def _allocation_changed(context, stock_exposure, defensive_exposure):
    old_stock = getattr(context, "current_stock_exposure", None)
    old_def = getattr(context, "current_defensive_exposure", None)
    if old_stock is None or old_def is None:
        return True
    return (
        abs(float(old_stock) - float(stock_exposure)) > 1e-8 or
        abs(float(old_def) - float(defensive_exposure)) > 1e-8
    )


def handle_data(context, data):
    current_date = get_current_date_from_engine(context, data)
    if current_date is None:
        return

    stock_exposure = _get_stock_exposure(context, current_date)
    defensive_exposure = _get_defensive_exposure(context, current_date)
    risk_on = bool(context.risk_on_by_date.get(current_date, True))

    is_rebalance_date = current_date in context.rebalance_dates
    allocation_changed = _allocation_changed(context, stock_exposure, defensive_exposure)

    # 非调仓日且趋势仓位没有变化时不重复下单。
    if (not is_rebalance_date) and (not allocation_changed):
        return

    # 调仓日更新目标股票池；非调仓日若只是趋势变化，则沿用上一期目标股票池，只调整仓位。
    if is_rebalance_date:
        today_signal = context.signal_by_date.get(current_date)
        if today_signal is None or len(today_signal) == 0:
            factor_targets = []
        else:
            factor_targets = sorted(today_signal["instrument"].astype(str).drop_duplicates().tolist())
        context.current_factor_targets = factor_targets
        context.current_target_date = current_date
    else:
        factor_targets = list(getattr(context, "current_factor_targets", []))

    defensive_targets = list(getattr(context, "defensive_bank_instruments", []))
    factor_target_set = set(factor_targets)
    defensive_target_set = set(defensive_targets)
    all_target_set = factor_target_set | defensive_target_set

    positions = get_positions_dict(context)
    holding_instruments = set()
    for ins, pos in positions.items():
        if position_amount(pos) > 0:
            holding_instruments.add(to_bigtrader_instrument(ins))

    for ins in sorted(holding_instruments - all_target_set):
        order_to_target_percent(context, ins, 0.0)

    defensive_weight = 0.0
    if len(defensive_targets) > 0:
        defensive_weight = float(defensive_exposure) / len(defensive_targets)
    for ins in sorted(defensive_targets):
        order_to_target_percent(context, ins, defensive_weight)

    factor_weight = 0.0
    if len(factor_targets) > 0:
        factor_weight = float(stock_exposure) / len(factor_targets)
    for ins in sorted(factor_targets):
        order_to_target_percent(context, ins, factor_weight)

    if is_rebalance_date and len(factor_targets) == 0:
        for ins in sorted(holding_instruments | defensive_target_set):
            order_to_target_percent(context, ins, 0.0)

    context.current_stock_exposure = stock_exposure
    context.current_defensive_exposure = defensive_exposure

    state_text = "风险开启/指数在30日均线上方" if risk_on else "风险关闭/指数跌破30日均线"
    if is_rebalance_date:
        print(
            f"{current_date} 调仓：{state_text}，因子股票 {len(factor_targets)} 只，"
            f"股票总仓位 {stock_exposure:.2%}，银行总仓位 {defensive_exposure:.2%}，"
            f"单只因子股 {factor_weight:.4f}，单只银行股 {defensive_weight:.4f}",
            flush=True,
        )
    elif allocation_changed:
        print(
            f"{current_date} 趋势仓位切换：{state_text}，沿用 {context.current_target_date} 目标池，"
            f"股票总仓位 {stock_exposure:.2%}，银行总仓位 {defensive_exposure:.2%}",
            flush=True,
        )


run_kwargs = dict(
    data=backtest_data,
    start_date=START_DATE,
    end_date=END_DATE,
    initialize=initialize,
    handle_data=handle_data,
    capital_base=CAPITAL_BASE,
    benchmark=BENCHMARK,
)

try:
    run_kwargs["market"] = bigtrader.Market.CN_STOCK
except Exception:
    pass

try:
    run_kwargs["frequency"] = bigtrader.Frequency.DAILY
except Exception:
    run_kwargs["frequency"] = "1d"

performance = bigtrader.run(**run_kwargs)

progress("BigTrader 回测完成")

try:
    display(performance.summary)
except Exception:
    display(performance)


## hml_r_std_5m 因子与 id2_std_3m 因子的比较

根据“图华泰因子复现/华泰波动率类因子/hml_r_std_5m因子/防御性+收益补偿2020-2026.png”、“华泰因子复现/华泰波动率类因子/hml_r_std_5m因子/防御性+收益补偿2022-2026.png”、“华泰因子复现/华泰波动率类因子/id2_std_3m因子/防御性+收益补偿回测2020-2026.png”、“华泰因子复现/华泰波动率类因子/id2_std_3m因子/防御性+收益补偿回测2022-2026.png”

从策略自身表现看，hml_r_std_5m 在后段时间表现更好，主要是因为它本质上是一个日内最大上涨波动率与最大下跌波动率之差，更像是在捕捉近期日内波动结构、资金博弈和短期情绪错配。你当前又是在小市值组内选因子值最低的股票，这类股票在 2024 年以后小微盘行情修复、流动性偏好回升、题材弹性增强的环境里更容易被放大收益，所以 hml_r_std_5m 在后半段出现了明显的净值加速。但它的问题也很明显：前期表现相对平淡，说明它的有效性更依赖特定市场状态，偏阶段性、偏交易型。

相比之下，id2_std_3m 是经过市场、规模、BP 三因子解释后的特质波动率残差标准差，再做市值行业中性化后，本质上更接近“剔除常见风险暴露后的个股噪声/特质风险”指标。选低值股票等于在小市值范围内选择相对更稳定、特质风险更低的股票，这种逻辑更像低波动/低特质风险异常，收益来源更稳健，因此在 2020—2026 这种更长周期里累计收益 287.34%、年化 24.43%，优于 hml_r_std_5m 的累计 230.84%、年化 21.16%。也就是说，id2_std_3m 的优势在于跨市场状态的稳定性更强，而 hml_r_std_5m 的优势在于近期行情中更贴合小市值弹性风格。

所以可以这样理解：hml_r_std_5m 更像一个阶段性增强因子，在小市值行情活跃、情绪修复、资金偏好弹性资产时表现更突出；id2_std_3m 更像一个长期稳健底仓因子，它不一定在每个阶段都最强，但在更长样本中更能持续筛掉高噪声、高特质风险股票。后续更合理的方向不是二选一，而是考虑把 id2_std_3m 作为主因子，hml_r_std_5m 作为近期风格增强因子，或者根据市场状态动态调整二者权重。